# Accuracy-first local fine-tuning (4 GB GPU)

This controlled continuation starts from the current best local v4 checkpoint,
keeps the model, losses, mining, and optimizer schedule, and changes the
positive-reference sampling from 80% vocal-separated to 20% vocal-separated
(80% original mix when available). Checkpoint selection uses all 300 gallery
tracks against the 773 validation queries. The test set is reserved for the
notebook's final report and must not be used to tune settings.

The run has its own name and checkpoint files; the existing model is preserved.
It is capped at 12 epochs with early stopping. Full-gallery validation is more
expensive than split-only validation on a 4 GB laptop GPU.


# Conformer-ByteHum: Local GPU 4 GB

Derived from `conformer_bytehum_qbh_kaggle.ipynb`. Run cells in order in a CUDA
PyTorch kernel. Dependencies: torch, numpy, pandas, matplotlib, tqdm, librosa,
soundfile, scipy, IPython, and typeguard<3. This notebook does not install packages.

Default: 2 songs x 2 queries = 4 triplets/microbatch, FP16 AMP, accumulation 16
(64 triplets/optimizer update), freeze first 3 blocks. Mining and BatchNorm still
operate on microbatches of 4; accumulation does not reproduce Kaggle batch-16
mining or BatchNorm. The architecture, CQT, losses and data split are preserved.

Run All trains from Stage 1 or resumes the local last checkpoint, selects best
on validation, then reports test local/global/fused MRR and Recall@1/5/10.
Test gallery contains test songs only (typically 45), not the full music database.
The default comparison loads the existing Kaggle v4 checkpoint under the same
test protocol. Use LOCAL_EVALUATE_ONLY=True to evaluate an existing local best.

Outputs: `02_stage2_humming_qbh/runs/local_4gb/`. CQT cache lives on disk with a
bounded RAM LRU. Resume is epoch-level; an interrupted epoch is run again.
Changing training settings requires a fresh run directory or matching checkpoint.


In [ ]:
from pathlib import Path
import os

# Edit these settings before Run All. Resume is from a completed epoch.
PROJECT_ROOT_OVERRIDE = None  # Example: Path(r"E:\HOCKY7\PBL6\code")
LOCAL_NUM_EPOCHS = 12
LOCAL_POSITIVE_VOCAL_RATIO = 0.20  # Compare 0.20, 0.50, and 0.80 in separate runs.
LOCAL_SONGS_PER_BATCH = 2
LOCAL_SAMPLES_PER_SONG = 2
LOCAL_ACCUMULATION_STEPS = 16
LOCAL_EVAL_BATCH_SIZE = 4
LOCAL_QUERY_BATCH_SIZE = 4
LOCAL_GALLERY_BLOCK_SIZE = 8
LOCAL_FREEZE_BLOCKS = 3
LOCAL_RAM_CACHE_MB = 256
LOCAL_RESUME = True
LOCAL_EVALUATE_ONLY = False
LOCAL_COMPARE_KAGGLE = True

if PROJECT_ROOT_OVERRIDE is None:
    roots = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next((root for root in roots
                         if (root / '02_stage2_humming_qbh').is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Set PROJECT_ROOT_OVERRIDE to the project root.')
else:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).resolve()
STAGE2_ROOT = PROJECT_ROOT / '02_stage2_humming_qbh'
INITIAL_LOCAL_CHECKPOINT = (STAGE2_ROOT / 'runs' / 'local_4gb' / 'checkpoints' /
                            'conformer_bytehum_full_d128_timegrid_lat_v4_local4gb_best.pth')
NUMBA_CACHE_DIR = STAGE2_ROOT / 'cache' / 'numba'
NUMBA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['NUMBA_CACHE_DIR'] = str(NUMBA_CACHE_DIR)
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['QBH_PREWARM_CQT'] = '0'
os.environ['QBH_VALIDATION_GALLERY'] = 'full'
os.environ['QBH_RESUME_TRAINING'] = '1' if LOCAL_RESUME else '0'
if min(LOCAL_SONGS_PER_BATCH, LOCAL_SAMPLES_PER_SONG, LOCAL_ACCUMULATION_STEPS,
       LOCAL_EVAL_BATCH_SIZE, LOCAL_QUERY_BATCH_SIZE, LOCAL_GALLERY_BLOCK_SIZE) < 1:
    raise ValueError('Batch and accumulation sizes must be positive.')
if LOCAL_SONGS_PER_BATCH < 2:
    raise ValueError('Mining requires at least two songs per microbatch.')
if not 0.0 <= LOCAL_POSITIVE_VOCAL_RATIO <= 1.0:
    raise ValueError('LOCAL_POSITIVE_VOCAL_RATIO must be between 0 and 1.')
print('Project:', PROJECT_ROOT)
print('Effective optimizer batch:', LOCAL_SONGS_PER_BATCH * LOCAL_SAMPLES_PER_SONG
      * LOCAL_ACCUMULATION_STEPS)


### 1.1 Cai dat va nap cac thu vien can thiet
Nap cac thu vien chuan cho xu ly am thanh (librosa, soundfile), tinh toan tensor (PyTorch) va tim kiem vector (FAISS). Tu dong cai dat bo sung neu chay tren Kaggle/Colab.

In [ ]:
from tqdm import tqdm
import os
import sys
import subprocess
import time
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display as ipd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from typing import Dict, List, Optional, Tuple

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules

if IS_KAGGLE or IS_COLAB:
    print("[*] Phat hien moi truong dam may (Kaggle / Colab). Kiem tra thu vien am thanh...")

try:
    import soundfile as sf
    import librosa
    import librosa.display
    import typeguard
    if not hasattr(typeguard, 'check_argument_types'):
        raise ImportError('CoverHunter requires typeguard<3')
except ImportError:
    if not (IS_KAGGLE or IS_COLAB):
        raise
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'librosa', 'typeguard<3'])
    sys.modules.pop('typeguard', None)
    import typeguard
    import soundfile as sf
    import librosa
    import librosa.display


### 1.2 Co dinh Seed va kiem tra phan cung GPU
Thiet lap seed ngau nhien (seed=42) de dam bao tinh tai lap cua ket qua thuc nghiem, dong thoi kiem tra thong so card do hoa (GPU VRAM).

In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(42)

if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('Select a CUDA-enabled PyTorch kernel before training.')
print(f"[+] Thiet bi tinh toan: {device}")
if device.type == 'cuda':
    print(f"[+] GPU Model        : {torch.cuda.get_device_name(0)}")
    print(f"[+] GPU VRAM         : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### 1.3 Dinh nghia ham dinh vi duong dan Dataset va Checkpoint
Tu dong giai nen datasets.zip tren Kaggle neu chua giai nen, sau do dinh vi thang vao cac thu muc con: dataset_split, 2_music_database, 4_vocal_separated va checkpoint Stage 1.

In [ ]:
def find_dataset_and_checkpoint():
    dataset_root = STAGE2_ROOT / 'datasets'
    candidates = [STAGE2_ROOT / 'checkpoints' / 'coverhunter_shs100k_pretrained.pth',
                  PROJECT_ROOT / '01_stage1_cover_song_csi' / 'checkpoints'
                  / 'coverhunter_shs100k_pretrained.pth']
    checkpoint = next((path for path in candidates if path.is_file()), None)
    if checkpoint is None:
        raise FileNotFoundError('Missing Stage-1 checkpoint: ' + str(candidates))
    split = dataset_root / 'dataset_split'
    for name in ('train', 'val', 'test'):
        if not (split / name).is_dir():
            raise FileNotFoundError(str(split / name))
    music = dataset_root / '2_music_database'
    vocals = dataset_root / '4_vocal_separated'
    return str(split), str(music), str(vocals if vocals.is_dir() else music), str(checkpoint)


### 1.4 Thiet lap thu muc luu tru ket qua va kiem tra duong dan du lieu
Thuc thi do tim duong dan, tao thu muc luu tru checkpoint (`checkpoints`) va kiem tra su ton tai cua cac thu muc du lieu.

In [ ]:
DATASET_SPLIT_DIR, MUSIC_DB_DIR, VOCAL_DIR, STAGE1_CKPT = find_dataset_and_checkpoint()
OUTPUT_DIR = str(STAGE2_ROOT / 'runs' / 'local_4gb')
CHECKPOINT_DIR = str(Path(OUTPUT_DIR) / 'checkpoints')
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
TRAIN_DIR, VAL_DIR, TEST_DIR = [str(Path(DATASET_SPLIT_DIR) / name) for name in ('train', 'val', 'test')]
print('Dataset:', DATASET_SPLIT_DIR)
print('Stage 1:', STAGE1_CKPT)
print('Outputs:', OUTPUT_DIR)


In [ ]:
# CoverHunter backbone is embedded to keep this notebook self-contained.
import logging
import math
from typing import Union
from typeguard import check_argument_types

class BasicModel(torch.nn.Module):
  """Minimal CoverHunter base class required by the embedded inference model."""

  def model_size(self) -> int:
    return sum(parameter.numel() for parameter in self.parameters())



class Linear(torch.nn.Module):
  def __init__(self, in_dim, out_dim, bias=True, w_init_gain='linear'):
    super(Linear, self).__init__()
    self.linear_layer = torch.nn.Linear(in_dim, out_dim, bias=bias)

    torch.nn.init.xavier_uniform_(
      self.linear_layer.weight,
      gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, x):
    return self.linear_layer(x)


class Conv1d(torch.nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=1, stride=1,
               padding=None, dilation=1, bias=True, w_init_gain='linear'):
    super(Conv1d, self).__init__()
    if padding is None:
      assert (kernel_size % 2 == 1)
      padding = int(dilation * (kernel_size - 1) / 2)

    self.conv = torch.nn.Conv1d(in_channels, out_channels,
                                kernel_size=kernel_size, stride=stride,
                                padding=padding, dilation=dilation,
                                bias=bias)

    torch.nn.init.xavier_uniform_(
      self.conv.weight, gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, signal):
    conv_signal = self.conv(signal)
    return conv_signal


class CenterLoss(nn.Module):
  """Center loss.

  Reference:
    A Discriminative Feature Learning Approach for Deep Face Recognition.

  Args:
      num_classes (int): number of classes.
      feat_dim (int): feature dimension.
  """

  def __init__(self, num_classes=10, feat_dim=2, use_gpu=True):
    super(CenterLoss, self).__init__()
    self.num_classes = num_classes
    self.feat_dim = feat_dim
    self.use_gpu = use_gpu
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    if self.use_gpu:
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).cuda())
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).to("cuda"))
    else:
      self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim))
    logging.info("CenterLoss: num_classes({}), feat_dim({})".format(
      num_classes, feat_dim))
    return

  def forward(self, x, labels):
    """
    Args:
        x: feature matrix with shape (batch_size, feat_dim).
        labels: ground truth labels with shape (batch_size).
    """
    batch_size = x.size(0)
    distmat = torch.pow(x, 2).sum(dim=1, keepdim=True). \
                expand(batch_size, self.num_classes) + \
              torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(
                self.num_classes, batch_size).t()
    distmat.addmm_(1, -2, x, self.centers.t())

    classes = torch.arange(self.num_classes).long()
    if self.use_gpu:
      classes = classes.cuda()
    labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
    mask = labels.eq(classes.expand(batch_size, self.num_classes))

    dist = distmat * mask.float()
    loss = dist.clamp(min=1e-12, max=1e+12).sum() / batch_size

    return loss


class FocalLoss(nn.Module):
  """ Focal Loss implement for https://arxiv.org/abs/1708.02002 """

  def __init__(self, gamma: float = 2., alpha: Optional[List] = None, num_cls: int = -1,
               reduction: str = 'mean'):

    super(FocalLoss, self).__init__()
    assert check_argument_types()
    if reduction not in ['mean', 'sum']:
      raise NotImplementedError(
        'Reduction {} not implemented.'.format(reduction))
    self._reduction = reduction
    self._alpha = alpha
    self._gamma = gamma
    if alpha is not None:
      assert len(alpha) <= num_cls, "{} != {}".format(len(alpha), num_cls)
      self._alpha = torch.tensor(self._alpha)
    self._eps = np.finfo(float).eps
    logging.info("Init Focal loss with gamma:{}".format(gamma))
    return

  def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """ compute focal loss for pred and label

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss
    """
    b = y_pred.size(0)
    y_pred_softmax = torch.nn.Softmax(dim=1)(y_pred) + self._eps
    ce = -torch.log(y_pred_softmax)
    ce = ce.gather(1, y_true.view(-1, 1))

    y_pred_softmax = y_pred_softmax.gather(1, y_true.view(-1, 1))
    weight = torch.pow(torch.sub(1., y_pred_softmax), self._gamma)

    if self._alpha is not None:
      self._alpha = self._alpha.to(y_pred.device)
      alpha = self._alpha.gather(0, y_true.view(-1))
      alpha = alpha.unsqueeze(1)
      alpha = alpha / torch.sum(alpha) * b
      weight = torch.mul(alpha, weight)
    fl_loss = torch.mul(weight, ce).squeeze(1)
    return self._reduce(fl_loss)

  def forward_with_onehot(self, y_pred: torch.Tensor,
                          y_true: torch.Tensor) -> torch.Tensor:
    """ Another implement for "forward" with onehot label matrix.

    It is not good because when ce_embed size is large, more memory will used.

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss

    """
    y_pred = torch.nn.Softmax(dim=1)(y_pred)
    y_true = torch.nn.functional.one_hot(y_true, self.num_cls)

    eps = np.finfo(float).eps
    y_pred = y_pred + eps
    ce = torch.mul(y_true, -torch.log(y_pred))
    weight = torch.mul(y_true, torch.pow(torch.sub(1., y_pred), self._gamma))
    fl_loss = torch.mul(self.alpha, torch.mul(weight, ce))

    fl_loss, _ = torch.max(fl_loss, dim=1)
    return self._reduce(fl_loss)

  def _reduce(self, x):
    if self._reduction == 'mean':
      return torch.mean(x)
    else:
      return torch.sum(x)


class HardTripletLoss(nn.Module):
  """Hard/Hardest Triplet Loss
    (pytorch implementation of https://omoindrot.github.io/triplet-loss)
    For each anchor, we get the hardest positive and hardest negative to form a triplet.
    """

  def __init__(self, margin=0.1):
    """ Args:
      margin: margin for triplet loss
    """
    super(HardTripletLoss, self).__init__()
    self._margin = margin
    return

  def forward(self, embeddings, labels):
    """
    Args:
        labels: labels of the batch, of size (batch_size,)
        embeddings: tensor of shape (batch_size, embed_dim)

    Returns:
        triplet_loss: scalar tensor containing the triplet loss
    """
    pairwise_dist = self._pairwise_distance(embeddings, squared=False)

    mask_anchor_positive = self._get_anchor_positive_triplet_mask(
      labels).float()
    valid_positive_dist = pairwise_dist * mask_anchor_positive
    hardest_positive_dist, _ = torch.max(valid_positive_dist, dim=1,
                                         keepdim=True)

    # Get the hardest negative pairs
    mask_negative = self._get_anchor_negative_triplet_mask(labels).float()
    max_negative_dist, _ = torch.max(pairwise_dist, dim=1, keepdim=True)
    negative_dist = pairwise_dist + max_negative_dist * (1.0 - mask_negative)
    hardest_negative_dist, _ = torch.min(negative_dist, dim=1, keepdim=True)

    # Combine biggest d(a, p) and smallest d(a, n) into final triplet loss
    triplet_loss = F.relu(
      hardest_positive_dist - hardest_negative_dist + self._margin)
    triplet_loss = torch.mean(triplet_loss)
    return triplet_loss

  @staticmethod
  def _pairwise_distance(x, squared=False, eps=1e-16):
    # Compute the 2D matrix of distances between all the embeddings.

    cor_mat = torch.matmul(x, x.t())
    norm_mat = cor_mat.diag()
    distances = norm_mat.unsqueeze(1) - 2 * cor_mat + norm_mat.unsqueeze(0)
    distances = F.relu(distances)

    if not squared:
      mask = torch.eq(distances, 0.0).float()
      distances = distances + mask * eps
      distances = torch.sqrt(distances)
      distances = distances * (1.0 - mask)
    return distances

  @staticmethod
  def _get_anchor_positive_triplet_mask(labels):
    """Return a 2D mask where mask[a, p] is True, if a and p are distinct and
       have same label.

    """
    # device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    device = labels.device
    indices_not_equal = torch.eye(labels.shape[0]).to(device).byte() ^ 1
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = indices_not_equal * labels_equal
    return mask

  @staticmethod
  def _get_anchor_negative_triplet_mask(labels):
    """Return a 2D mask where mask[a, n] is True iff a and n have distinct labels.
    """
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = labels_equal ^ 1
    return mask


def subsequent_chunk_mask(
    size: int,
    chunk_size: int,
    num_left_chunks: int = -1,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
  """Create mask for subsequent steps (size, size) with chunk size,
       this is for streaming encoder

    Args:
        size (int): size of mask
        chunk_size (int): size of chunk
        num_left_chunks (int): number of left chunks
            <0: use full chunk
            >=0: use num_left_chunks
        device (torch.device): "cpu" or "cuda" or torch.Tensor.device

    Returns:
        torch.Tensor: mask

    Examples:
        >>> subsequent_chunk_mask(4, 2)
        [[1, 1, 0, 0],
         [1, 1, 0, 0],
         [1, 1, 1, 1],
         [1, 1, 1, 1]]
    """
  ret = torch.zeros(size, size, device=device, dtype=torch.bool)
  for i in range(size):
    if num_left_chunks < 0:
      start = 0
    else:
      start = max((i // chunk_size - num_left_chunks) * chunk_size, 0)
    ending = min((i // chunk_size + 1) * chunk_size, size)
    ret[i, start:ending] = True
  return ret


def add_optional_chunk_mask(xs: torch.Tensor, masks: torch.Tensor,
                            use_dynamic_chunk: bool,
                            use_dynamic_left_chunk: bool,
                            decoding_chunk_size: int, static_chunk_size: int,
                            num_decoding_left_chunks: int):
  """ Apply optional mask for encoder.

    Args:
        masks:
        xs (torch.Tensor): padded input, (B, L, D), L for max length
        mask (torch.Tensor): mask for xs, (B, 1, L)
        use_dynamic_chunk (bool): whether to use dynamic chunk or not
        use_dynamic_left_chunk (bool): whether to use dynamic left chunk for
            training.
        decoding_chunk_size (int): decoding chunk size for dynamic chunk, it's
            0: default for training, use random dynamic chunk.
            <0: for decoding, use full chunk.
            >0: for decoding, use fixed chunk size as set.
        static_chunk_size (int): chunk size for static chunk training/decoding
            if it's greater than 0, if use_dynamic_chunk is true,
            this parameter will be ignored
        num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
            >=0: use num_decoding_left_chunks
            <0: use all left chunks

    Returns:
        torch.Tensor: chunk mask of the input xs.
    """
  # Whether to use chunk mask or not
  if use_dynamic_chunk:
    max_len = xs.size(1)
    if decoding_chunk_size < 0:
      chunk_size = max_len
      num_left_chunks = -1
    elif decoding_chunk_size > 0:
      chunk_size = decoding_chunk_size
      num_left_chunks = num_decoding_left_chunks
    else:
      # chunk size is either [1, 25] or full context(max_len).
      # Since we use 4 times subsampling and allow up to 1s(100 frames)
      # delay, the maximum frame is 100 / 4 = 25.
      chunk_size = torch.randint(1, max_len, (1,)).item()
      num_left_chunks = -1
      if chunk_size > max_len // 2:
        chunk_size = max_len
      else:
        chunk_size = chunk_size % 25 + 1
        if use_dynamic_left_chunk:
          max_left_chunks = (max_len - 1) // chunk_size
          num_left_chunks = torch.randint(0, max_left_chunks,
                                          (1,)).item()
    chunk_masks = subsequent_chunk_mask(xs.size(1), chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  elif static_chunk_size > 0:
    num_left_chunks = num_decoding_left_chunks
    chunk_masks = subsequent_chunk_mask(xs.size(1), static_chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  else:
    chunk_masks = masks
  return chunk_masks


def make_pad_mask(lengths: torch.Tensor, max_len: int = 0) -> torch.Tensor:
  """Make mask tensor containing indices of padded part.

    See description of make_non_pad_mask.

    Args:
        lengths (torch.Tensor): Batch of lengths (B,).
        max_len:

    Returns:
        torch.Tensor: Mask tensor containing indices of padded part.

    Examples:
        >>> make_pad_mask(torch.tensor([5, 3, 2]))
        masks = [[0, 0, 0, 0 ,0],
                 [0, 0, 0, 1, 1],
                 [0, 0, 1, 1, 1]]
    """
  batch_size = lengths.size(0)
  max_len = max_len if max_len > 0 else lengths.max().item()
  seq_range = torch.arange(0,
                           max_len,
                           dtype=torch.int64,
                           device=lengths.device)
  seq_range_expand = seq_range.unsqueeze(0).expand(batch_size, max_len)
  seq_length_expand = lengths.unsqueeze(-1)
  mask = seq_range_expand >= seq_length_expand
  return mask


class Swish(torch.nn.Module):
  """Construct an Swish object."""

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """Return Swish activation function."""
    return x * torch.sigmoid(x)


def get_activation(act):
  """Return activation function."""
  activation_funcs = {
    "hardtanh": torch.nn.Hardtanh,
    "tanh": torch.nn.Tanh,
    "relu": torch.nn.ReLU,
    "selu": torch.nn.SELU,
    "swish": getattr(torch.nn, "SiLU", Swish),
    "gelu": torch.nn.GELU
  }
  return activation_funcs[act]()


class PositionalEncoding(torch.nn.Module):
  """Positional encoding.

    :param int d_model: embedding dim
    :param float dropout_rate: dropout rate
    :param int max_len: maximum input length

    PE(pos, 2i)   = sin(pos/(10000^(2i/dmodel)))
    PE(pos, 2i+1) = cos(pos/(10000^(2i/dmodel)))
    """

  def __init__(self,
               d_model: int,
               dropout_rate: float,
               max_len: int = 5000,
               reverse: bool = False):
    """Construct an PositionalEncoding object."""
    super().__init__()
    self.d_model = d_model
    self.xscale = math.sqrt(self.d_model)
    self.dropout = torch.nn.Dropout(p=dropout_rate)
    self.max_len = max_len

    self.pe = torch.zeros(self.max_len, self.d_model)
    position = torch.arange(0, self.max_len,
                            dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
      torch.arange(0, self.d_model, 2, dtype=torch.float32) *
      -(math.log(10000.0) / self.d_model))
    self.pe[:, 0::2] = torch.sin(position * div_term)
    self.pe[:, 1::2] = torch.cos(position * div_term)
    self.pe = self.pe.unsqueeze(0)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Add positional encoding.

        Args:
            x (torch.Tensor): Input. Its shape is (batch, time, ...)
            offset (int, torch.tensor): position offset

        Returns:
            torch.Tensor: Encoded tensor. Its shape is (batch, time, ...)
            torch.Tensor: for compatibility to RelPositionalEncoding
        """

    self.pe = self.pe.to(x.device)
    pos_emb = self.position_encoding(offset, x.size(1), False)
    x = x * self.xscale + pos_emb
    return self.dropout(x), self.dropout(pos_emb)

  def position_encoding(self, offset: Union[int, torch.Tensor], size: int,
                        apply_dropout: bool = True) -> torch.Tensor:
    """ For getting encoding in a streaming fashion

        Attention!!!!!
        we apply dropout only once at the whole utterance level in a none
        streaming way, but will call this function several times with
        increasing input size in a streaming scenario, so the dropout will
        be applied several times.

        Args:
            apply_dropout:
            offset (int or torch.tensor): start offset
            size (int): required size of position encoding

        Returns:
            torch.Tensor: Corresponding encoding
        """
    # How to subscript a Union type:
    #   https://github.com/pytorch/pytorch/issues/69434
    if isinstance(offset, int):
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    elif isinstance(offset, torch.Tensor) and offset.dim() == 0:  # scalar
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    else:  # for batched streaming decoding on GPU
      assert torch.max(offset) + size < self.max_len
      index = offset.unsqueeze(1) + torch.arange(0, size).to(offset.device)  # B X T
      flag = index > 0
      # remove negative offset
      index = index * flag
      pos_emb = F.embedding(index, self.pe[0])  # B X T X d_model

    if apply_dropout:
      pos_emb = self.dropout(pos_emb)
    return pos_emb


class RelPositionalEncoding(PositionalEncoding):
  """Relative positional encoding module.
    See : Appendix B in https://arxiv.org/abs/1901.02860
    Args:
        d_model (int): Embedding dimension.
        dropout_rate (float): Dropout rate.
        max_len (int): Maximum input length.
    """

  def __init__(self, d_model: int, dropout_rate: float, max_len: int = 5000):
    """Initialize class."""
    super().__init__(d_model, dropout_rate, max_len, reverse=True)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute positional encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, `*`).
        Returns:
            torch.Tensor: Encoded tensor (batch, time, `*`).
            torch.Tensor: Positional embedding tensor (1, time, `*`).
        """
    self.pe = self.pe.to(x.device)
    x = x * self.xscale
    pos_emb = self.position_encoding(offset, x.size(1), False)
    return self.dropout(x), self.dropout(pos_emb)


class NoPositionalEncoding(torch.nn.Module):
  """ No position encoding
    """

  def __init__(self, d_model: int, dropout_rate: float):
    super().__init__()
    self.d_model = d_model
    self.dropout = torch.nn.Dropout(p=dropout_rate)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """ Just return zero vector for interface compatibility
        """
    pos_emb = torch.zeros(1, x.size(1), self.d_model).to(x.device)
    return self.dropout(x), pos_emb

  def position_encoding(
      self, offset: Union[int, torch.Tensor], size: int) -> torch.Tensor:
    return torch.zeros(1, size, self.d_model)


class BaseSubsampling(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.right_context = 0
    self.subsampling_rate = 1

  def position_encoding(self, offset: Union[int, torch.Tensor],
                        size: int) -> torch.Tensor:
    return self.pos_enc.position_encoding(offset, size)


class LinearNoSubsampling(BaseSubsampling):
  """Linear transform the input without subsampling

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an linear object."""
    super().__init__()
    self.out = torch.nn.Sequential(
      torch.nn.Linear(idim, odim),
      torch.nn.LayerNorm(odim, eps=1e-5),
      torch.nn.Dropout(dropout_rate),
    )
    self.pos_enc = pos_enc_class
    self.right_context = 0
    self.subsampling_rate = 1

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Input x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: linear input tensor (#batch, time', odim),
                where time' = time .
            torch.Tensor: linear input mask (#batch, 1, time'),
                where time' = time .

        """
    x = self.out(x)
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask


class Conv2dSubsampling4(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/4 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling4 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.out = torch.nn.Sequential(
      torch.nn.Linear(odim * (((idim - 1) // 2 - 1) // 2), odim))
    self.pos_enc = pos_enc_class
    # The right context for every conv layer is computed by:
    # (kernel_size - 1) * frame_rate_of_this_layer
    self.subsampling_rate = 4
    # 6 = (3 - 1) * 1 + (3 - 1) * 2
    self.right_context = 6

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 4.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 4.
            torch.Tensor: positional encoding

        """
    x = x.unsqueeze(1)  # (b, c=1, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.out(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2]


class Conv2dSubsampling8(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/8 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling8 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.linear = torch.nn.Linear(
      odim * ((((idim - 1) // 2 - 1) // 2 - 1) // 2), odim)
    self.pos_enc = pos_enc_class
    self.subsampling_rate = 8
    # 14 = (3 - 1) * 1 + (3 - 1) * 2 + (3 - 1) * 4
    self.right_context = 14

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 8.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 8.
            torch.Tensor: positional encoding
        """
    x = x.unsqueeze(1)  # (b, c, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.linear(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2][:, :, :-2:2]


class MultiHeadedAttention(nn.Module):
  """Multi-Head Attention layer.

    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, n_head: int, n_feat: int, dropout_rate: float):
    """Construct an MultiHeadedAttention object."""
    super().__init__()
    assert n_feat % n_head == 0
    # We assume d_v always equals d_k
    self.d_k = n_feat // n_head
    self.h = n_head
    self.linear_q = nn.Linear(n_feat, n_feat)
    self.linear_k = nn.Linear(n_feat, n_feat)
    self.linear_v = nn.Linear(n_feat, n_feat)
    self.linear_out = nn.Linear(n_feat, n_feat)
    self.dropout = nn.Dropout(p=dropout_rate)

  def forward_qkv(
      self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Transform query, key and value.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).

        Returns:
            torch.Tensor: Transformed query tensor, size
                (#batch, n_head, time1, d_k).
            torch.Tensor: Transformed key tensor, size
                (#batch, n_head, time2, d_k).
            torch.Tensor: Transformed value tensor, size
                (#batch, n_head, time2, d_k).

        """
    n_batch = query.size(0)
    q = self.linear_q(query).view(n_batch, -1, self.h, self.d_k)
    k = self.linear_k(key).view(n_batch, -1, self.h, self.d_k)
    v = self.linear_v(value).view(n_batch, -1, self.h, self.d_k)
    q = q.transpose(1, 2)  # (batch, head, time1, d_k)
    k = k.transpose(1, 2)  # (batch, head, time2, d_k)
    v = v.transpose(1, 2)  # (batch, head, time2, d_k)

    return q, k, v

  def forward_attention(
      self, value: torch.Tensor, scores: torch.Tensor,
      mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool)
  ) -> torch.Tensor:
    """Compute attention context vector.

        Args:
            value (torch.Tensor): Transformed value, size
                (#batch, n_head, time2, d_k).
            scores (torch.Tensor): Attention score, size
                (#batch, n_head, time1, time2).
            mask (torch.Tensor): Mask, size (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.

        Returns:
            torch.Tensor: Transformed value (#batch, time1, d_model)
                weighted by the attention score (#batch, time1, time2).

        """
    n_batch = value.size(0)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be True?
    #   1. onnx(16/4) [WHY? Because we feed real cache & real mask for the
    #           1st chunk to ease the onnx export.]
    #   2. pytorch training
    if mask.size(2) > 0:  # time2 > 0
      mask = mask.unsqueeze(1).eq(0)  # (batch, 1, *, time2)
      # For last chunk, time2 might be larger than scores.size(-1)
      mask = mask[:, :, :, :scores.size(-1)]  # (batch, 1, *, time2)
      scores = scores.masked_fill(mask, -float('inf'))
      attn = torch.softmax(scores, dim=-1).masked_fill(
        mask, 0.0)  # (batch, head, time1, time2)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be False?
    #   1. onnx(16/-1, -1/-1, 16/0)
    #   2. jit (16/-1, -1/-1, 16/0, 16/4)
    else:
      attn = torch.softmax(scores, dim=-1)  # (batch, head, time1, time2)

    p_attn = self.dropout(attn)
    x = torch.matmul(p_attn, value)  # (batch, head, time1, d_k)
    x = (x.transpose(1, 2).contiguous().view(n_batch, -1,
                                             self.h * self.d_k)
         )  # (batch, time1, d_model)

    return self.linear_out(x)  # (batch, time1, d_model)

  def forward(self, query: torch.Tensor, key: torch.Tensor,
              value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot product attention.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2).
                1.When applying cross attention between decoder and encoder,
                the batch padding mask for input is in (#batch, 1, T) shape.
                2.When applying self attention of encoder,
                the mask is in (#batch, T, T)  shape.
                3.When applying self attention of decoder,
                the mask is in (#batch, L, L)  shape.
                4.If the different position in decoder see different block
                of the encoder, such as Mocha, the passed in mask could be
                in (#batch, L, T) shape. But there is no such case in current
                Wenet.
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`


        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`

        """
    q, k, v = self.forward_qkv(query, key, value)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
    return self.forward_attention(v, scores, mask), new_cache


class RelPositionMultiHeadedAttention(MultiHeadedAttention):
  """Multi-Head Attention layer with relative position encoding.
    Paper: https://arxiv.org/abs/1901.02860
    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.
    """

  def __init__(self, n_head, n_feat, dropout_rate):
    """Construct an RelPositionMultiHeadedAttention object."""
    super().__init__(n_head, n_feat, dropout_rate)
    # linear transformation for positional encoding
    self.linear_pos = nn.Linear(n_feat, n_feat, bias=False)
    # these two learnable bias are used in matrix c and matrix d
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    self.pos_bias_u = nn.Parameter(torch.Tensor(self.h, self.d_k))
    self.pos_bias_v = nn.Parameter(torch.Tensor(self.h, self.d_k))
    torch.nn.init.xavier_uniform_(self.pos_bias_u)
    torch.nn.init.xavier_uniform_(self.pos_bias_v)

  def rel_shift(self, x, zero_triu: bool = False):
    """Compute relative positinal encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, size).
            zero_triu (bool): If true, return the lower triangular part of
                the matrix.
        Returns:
            torch.Tensor: Output tensor.
        """

    zero_pad = torch.zeros((x.size()[0], x.size()[1], x.size()[2], 1),
                           device=x.device,
                           dtype=x.dtype)
    x_padded = torch.cat([zero_pad, x], dim=-1)

    x_padded = x_padded.view(x.size()[0],
                             x.size()[1],
                             x.size(3) + 1, x.size(2))
    x = x_padded[:, :, 1:].view_as(x)

    if zero_triu:
      ones = torch.ones((x.size(2), x.size(3)))
      x = x * torch.tril(ones, x.size(3) - x.size(2))[None, None, :, :]

    return x

  def forward(self, query: torch.Tensor,
              key: torch.Tensor, value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute 'Scaled Dot Product Attention' with rel. positional encoding.
        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): Positional embedding tensor
                (#batch, time2, size).
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        """
    q, k, v = self.forward_qkv(query, key, value)
    q = q.transpose(1, 2)  # (batch, time1, head, d_k)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    n_batch_pos = pos_emb.size(0)
    p = self.linear_pos(pos_emb).view(n_batch_pos, -1, self.h, self.d_k)
    p = p.transpose(1, 2)  # (batch, head, time1, d_k)

    # (batch, head, time1, d_k)
    q_with_bias_u = (q + self.pos_bias_u).transpose(1, 2)
    # (batch, head, time1, d_k)
    q_with_bias_v = (q + self.pos_bias_v).transpose(1, 2)

    # compute attention score
    # first compute matrix a and matrix c
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    # (batch, head, time1, time2)
    matrix_ac = torch.matmul(q_with_bias_u, k.transpose(-2, -1))

    # compute matrix b and matrix d
    # (batch, head, time1, time2)
    matrix_bd = torch.matmul(q_with_bias_v, p.transpose(-2, -1))
    # Remove rel_shift since it is useless in speech recognition,
    # and it requires special attention for streaming.
    # matrix_bd = self.rel_shift(matrix_bd)

    scores = (matrix_ac + matrix_bd) / math.sqrt(
      self.d_k)  # (batch, head, time1, time2)

    return self.forward_attention(v, scores, mask), new_cache


class ConvolutionModule(nn.Module):
  """ConvolutionModule in Conformer model."""

  def __init__(self,
               channels: int,
               kernel_size: int = 15,
               activation: nn.Module = nn.ReLU(),
               norm: str = "batch_norm",
               causal: bool = False,
               bias: bool = True):
    """Construct an ConvolutionModule object.
        Args:
            channels (int): The number of channels of conv layers.
            kernel_size (int): Kernel size of conv layers.
            causal (int): Whether use causal convolution or not
        """
    assert check_argument_types()
    super().__init__()

    self.pointwise_conv1 = nn.Conv1d(
      channels,
      2 * channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    # self.lorder is used to distinguish if it's a causal convolution,
    # if self.lorder > 0: it's a causal convolution, the input will be
    #    padded with self.lorder frames on the left in forward.
    # else: it's a symmetrical convolution
    if causal:
      padding = 0
      self.lorder = kernel_size - 1
    else:
      # kernel_size should be an odd number for none causal convolution
      assert (kernel_size - 1) % 2 == 0
      padding = (kernel_size - 1) // 2
      self.lorder = 0
    self.depthwise_conv = nn.Conv1d(
      channels,
      channels,
      kernel_size,
      stride=1,
      padding=padding,
      groups=channels,
      bias=bias,
    )

    assert norm in ['batch_norm', 'layer_norm']
    if norm == "batch_norm":
      self.use_layer_norm = False
      self.norm = nn.BatchNorm1d(channels)
    else:
      self.use_layer_norm = True
      self.norm = nn.LayerNorm(channels)

    self.pointwise_conv2 = nn.Conv1d(
      channels,
      channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    self.activation = activation

  def forward(
      self,
      x: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      cache: torch.Tensor = torch.zeros((0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute convolution module.
        Args:
            x (torch.Tensor): Input tensor (#batch, time, channels).
            mask_pad (torch.Tensor): used for batch padding (#batch, 1, time),
                (0, 0, 0) means fake mask.
            cache (torch.Tensor): left context cache, it is only
                used in causal convolution (#batch, channels, cache_t),
                (0, 0, 0) meas fake cache.
        Returns:
            torch.Tensor: Output tensor (#batch, time, channels).
        """
    # exchange the temporal dimension and the feature dimension
    x = x.transpose(1, 2)  # (#batch, channels, time)

    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    if self.lorder > 0:
      if cache.size(2) == 0:  # cache_t == 0
        x = nn.functional.pad(x, (self.lorder, 0), 'constant', 0.0)
      else:
        assert cache.size(0) == x.size(0)  # equal batch
        assert cache.size(1) == x.size(1)  # equal channel
        x = torch.cat((cache, x), dim=2)
      assert (x.size(2) > self.lorder)
      new_cache = x[:, :, -self.lorder:]
    else:
      # It's better we just return None if no cache is required,
      # However, for JIT export, here we just fake one tensor instead of
      # None.
      new_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)

    # GLU mechanism
    x = self.pointwise_conv1(x)  # (batch, 2*channel, dim)
    x = nn.functional.glu(x, dim=1)  # (batch, channel, dim)

    # 1D Depthwise Conv
    x = self.depthwise_conv(x)
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.activation(self.norm(x))
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.pointwise_conv2(x)
    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    return x.transpose(1, 2), new_cache


class PositionwiseFeedForward(torch.nn.Module):
  """Positionwise feed forward layer.

    FeedForward are appied on each position of the sequence.
    The output dim is same with the input dim.

    Args:
        idim (int): Input dimenstion.
        hidden_units (int): The number of hidden units.
        dropout_rate (float): Dropout rate.
        activation (torch.nn.Module): Activation function
    """

  def __init__(self,
               idim: int,
               hidden_units: int,
               dropout_rate: float,
               activation: torch.nn.Module = torch.nn.ReLU()):
    """Construct a PositionwiseFeedForward object."""
    super(PositionwiseFeedForward, self).__init__()
    self.w_1 = torch.nn.Linear(idim, hidden_units)
    self.activation = activation
    self.dropout = torch.nn.Dropout(dropout_rate)
    self.w_2 = torch.nn.Linear(hidden_units, idim)

  def forward(self, xs: torch.Tensor) -> torch.Tensor:
    """Forward function.

        Args:
            xs: input tensor (B, L, D)
        Returns:
            output tensor, (B, L, D)
        """
    return self.w_2(self.dropout(self.activation(self.w_1(xs))))


class ConformerEncoderLayer(torch.nn.Module):
  """Encoder layer module.
    Args:
        size (int): Input dimension.
        self_attn (torch.nn.Module): Self-attention module instance.
            `MultiHeadedAttention` or `RelPositionMultiHeadedAttention`
            instance can be used as the argument.
        feed_forward (torch.nn.Module): Feed-forward module instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        feed_forward_macaron (torch.nn.Module): Additional feed-forward module
             instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        conv_module (torch.nn.Module): Convolution module instance.
            `ConvlutionModule` instance can be used as the argument.
        dropout_rate (float): Dropout rate.
        normalize_before (bool):
            True: use layer_norm before each sub-block.
            False: use layer_norm after each sub-block.
        concat_after (bool): Whether to concat attention layer's input and
            output.
            True: x -> x + linear(concat(x, att(x)))
            False: x -> x + att(x)
    """

  def __init__(
      self,
      size: int,
      self_attn: torch.nn.Module,
      feed_forward: Optional[nn.Module] = None,
      feed_forward_macaron: Optional[nn.Module] = None,
      conv_module: Optional[nn.Module] = None,
      dropout_rate: float = 0.1,
      normalize_before: bool = True,
      concat_after: bool = False,
  ):
    """Construct an EncoderLayer object."""
    super().__init__()
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.feed_forward_macaron = feed_forward_macaron
    self.conv_module = conv_module
    self.norm_ff = nn.LayerNorm(size, eps=1e-5)  # for the FNN module
    self.norm_mha = nn.LayerNorm(size, eps=1e-5)  # for the MHA module
    if feed_forward_macaron is not None:
      self.norm_ff_macaron = nn.LayerNorm(size, eps=1e-5)
      self.ff_scale = 0.5
    else:
      self.ff_scale = 1.0
    if self.conv_module is not None:
      self.norm_conv = nn.LayerNorm(size,
                                    eps=1e-5)  # for the CNN module
      self.norm_final = nn.LayerNorm(
        size, eps=1e-5)  # for the final output of the block
    self.dropout = nn.Dropout(dropout_rate)
    self.size = size
    self.normalize_before = normalize_before
    self.concat_after = concat_after
    if self.concat_after:
      self.concat_linear = nn.Linear(size + size, size)
    else:
      self.concat_linear = nn.Identity()

  def forward(
      self,
      x: torch.Tensor,
      mask: torch.Tensor,
      pos_emb: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
      cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute encoded features.

        Args:
            x (torch.Tensor): (#batch, time, size)
            mask (torch.Tensor): Mask tensor for the input (#batch, time，time),
                (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): positional encoding, must not be None
                for ConformerEncoderLayer.
            mask_pad (torch.Tensor): batch padding mask used for conv module.
                (#batch, 1，time), (0, 0, 0) means fake mask.
            att_cache (torch.Tensor): Cache tensor of the KEY & VALUE
                (#batch=1, head, cache_t1, d_k * 2), head * d_k == size.
            cnn_cache (torch.Tensor): Convolution cache in conformer layer
                (#batch=1, size, cache_t2)
        Returns:
            torch.Tensor: Output tensor (#batch, time, size).
            torch.Tensor: Mask tensor (#batch, time, time).
            torch.Tensor: att_cache tensor,
                (#batch=1, head, cache_t1 + time, d_k * 2).
            torch.Tensor: cnn_cahce tensor (#batch, size, cache_t2).
        """

    # whether to use macaron style
    if self.feed_forward_macaron is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_ff_macaron(x)
      x = residual + self.ff_scale * self.dropout(
        self.feed_forward_macaron(x))
      if not self.normalize_before:
        x = self.norm_ff_macaron(x)

    # multi-headed self-attention module
    residual = x
    if self.normalize_before:
      x = self.norm_mha(x)

    x_att, new_att_cache = self.self_attn(
      x, x, x, mask, pos_emb, att_cache)
    if self.concat_after:
      x_concat = torch.cat((x, x_att), dim=-1)
      x = residual + self.concat_linear(x_concat)
    else:
      x = residual + self.dropout(x_att)
    if not self.normalize_before:
      x = self.norm_mha(x)

    # convolution module
    # Fake new cnn cache here, and then change it in conv_module
    new_cnn_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)
    if self.conv_module is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_conv(x)
      x, new_cnn_cache = self.conv_module(x, mask_pad, cnn_cache)
      x = residual + self.dropout(x)

      if not self.normalize_before:
        x = self.norm_conv(x)

    # feed forward module
    residual = x
    if self.normalize_before:
      x = self.norm_ff(x)

    x = residual + self.ff_scale * self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x = self.norm_ff(x)

    if self.conv_module is not None:
      x = self.norm_final(x)

    return x, mask, new_att_cache, new_cnn_cache


class ConformerEncoder(torch.nn.Module):
  def __init__(
      self,
      input_size: int,
      output_size: int,
      attention_heads: int = 4,
      linear_units: int = 2048,
      num_blocks: int = 6,
      dropout_rate: float = 0.1,
      positional_dropout_rate: float = 0.1,
      attention_dropout_rate: float = 0.0,
      input_layer: str = "conv2d",
      pos_enc_layer_type: str = "rel_pos",
      normalize_before: bool = True,
      concat_after: bool = False,
      static_chunk_size: int = 0,
      use_dynamic_chunk: bool = False,
      global_cmvn: Optional[torch.nn.Module] = None,
      use_dynamic_left_chunk: bool = False,
      positionwise_conv_kernel_size: int = 1,
      macaron_style: bool = True,
      selfattention_layer_type: str = "rel_selfattn",
      activation_type: str = "swish",
      use_cnn_module: bool = True,
      cnn_module_kernel: int = 15,
      causal: bool = False,
      cnn_module_norm: str = "batch_norm",
  ):
    """
        Args:
            input_size (int): input dim
            output_size (int): dimension of attention
            attention_heads (int): the number of heads of multi head attention
            linear_units (int): the hidden units number of position-wise feed
                forward
            num_blocks (int): the number of decoder blocks
            dropout_rate (float): dropout rate
            attention_dropout_rate (float): dropout rate in attention
            positional_dropout_rate (float): dropout rate after adding
                positional encoding
            input_layer (str): input layer type.
                optional [linear, conv2d, conv2d6, conv2d8]
            pos_enc_layer_type (str): Encoder positional encoding layer type.
                opitonal [abs_pos, scaled_abs_pos, rel_pos, no_pos]
            normalize_before (bool):
                True: use layer_norm before each sub-block of a layer.
                False: use layer_norm after each sub-block of a layer.
            concat_after (bool): whether to concat attention layer's input
                and output.
                True: x -> x + linear(concat(x, att(x)))
                False: x -> x + att(x)
            static_chunk_size (int): chunk size for static chunk training and
                decoding
            use_dynamic_chunk (bool): whether use dynamic chunk size for
                training or not, You can only use fixed chunk(chunk_size > 0)
                or dyanmic chunk size(use_dynamic_chunk = True)
            global_cmvn (Optional[torch.nn.Module]): Optional GlobalCMVN module
            use_dynamic_left_chunk (bool): whether use dynamic left chunk in
                dynamic chunk training
        """
    assert check_argument_types()
    logging.info("Init Conformer layers::")
    super().__init__()
    self._output_size = output_size

    if pos_enc_layer_type == "abs_pos":
      pos_enc_class = PositionalEncoding
    elif pos_enc_layer_type == "rel_pos":
      pos_enc_class = RelPositionalEncoding
    elif pos_enc_layer_type == "no_pos":
      pos_enc_class = NoPositionalEncoding
    else:
      raise ValueError("unknown pos_enc_layer: " + pos_enc_layer_type)
    logging.info("Pos layers - {}".format(pos_enc_layer_type))

    if input_layer == "linear":
      subsampling_class = LinearNoSubsampling
    elif input_layer == "conv2d":
      subsampling_class = Conv2dSubsampling4
    elif input_layer == "conv2d8":
      subsampling_class = Conv2dSubsampling8
    else:
      raise ValueError("unknown input_layer: " + input_layer)
    logging.info("Input layers - {}".format(input_layer))

    self.global_cmvn = global_cmvn
    self.embed = subsampling_class(
      input_size,
      output_size,
      dropout_rate,
      pos_enc_class(output_size, positional_dropout_rate),
    )

    self.normalize_before = normalize_before
    self.after_norm = torch.nn.LayerNorm(output_size, eps=1e-5)
    self.static_chunk_size = static_chunk_size
    self.use_dynamic_chunk = use_dynamic_chunk
    self.use_dynamic_left_chunk = use_dynamic_left_chunk

    activation = get_activation(activation_type)

    # self-attention module definition
    if pos_enc_layer_type != "rel_pos":
      encoder_selfattn_layer = MultiHeadedAttention
    else:
      encoder_selfattn_layer = RelPositionMultiHeadedAttention
    encoder_selfattn_layer_args = (
      attention_heads,
      output_size,
      attention_dropout_rate,
    )
    logging.info("self attn args: {}".format(encoder_selfattn_layer_args))

    # feed-forward module definition
    positionwise_layer = PositionwiseFeedForward
    positionwise_layer_args = (
      output_size,
      linear_units,
      dropout_rate,
      activation,
    )
    logging.info("position-wise args: {}".format(positionwise_layer_args))

    # convolution module definition
    convolution_layer = ConvolutionModule
    convolution_layer_args = (output_size, cnn_module_kernel, activation,
                              cnn_module_norm, causal)

    self.encoders = torch.nn.ModuleList([
      ConformerEncoderLayer(
        output_size,
        encoder_selfattn_layer(*encoder_selfattn_layer_args),
        positionwise_layer(*positionwise_layer_args),
        positionwise_layer(
          *positionwise_layer_args) if macaron_style else None,
        convolution_layer(
          *convolution_layer_args) if use_cnn_module else None,
        dropout_rate,
        normalize_before,
        concat_after,
      ) for _ in range(num_blocks)
    ])
    return

  def output_size(self) -> int:
    return self._output_size

  def forward(
      self,
      xs: torch.Tensor,
      xs_lens: torch.Tensor,
      decoding_chunk_size: int = -1,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Embed positions in tensor.

        Args:
            xs: padded input tensor (B, T, D)
            xs_lens: input length (B)
            decoding_chunk_size: decoding chunk size for dynamic chunk
                0: default for training, use random dynamic chunk.
                <0: for decoding, use full chunk.
                >0: for decoding, use fixed chunk size as set.
            num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
                >=0: use num_decoding_left_chunks
                <0: use all left chunks
        Returns:
            encoder output tensor xs, and subsampled masks
            xs: padded output tensor (B, T' ~= T/subsample_rate, D)
            masks: torch.Tensor batch padding mask after subsample
                (B, 1, T' ~= T/subsample_rate)
        """
    T = xs.size(1)
    masks = ~make_pad_mask(xs_lens, T).unsqueeze(1)  # (B, 1, T)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    xs, pos_emb, masks = self.embed(xs, masks)
    mask_pad = masks  # (B, 1, T/subsample_rate)
    chunk_masks = add_optional_chunk_mask(xs, masks,
                                          self.use_dynamic_chunk,
                                          self.use_dynamic_left_chunk,
                                          decoding_chunk_size,
                                          self.static_chunk_size,
                                          num_decoding_left_chunks)
    for layer in self.encoders:
      xs, chunk_masks, _, _ = layer(xs, chunk_masks, pos_emb, mask_pad)
    if self.normalize_before:
      xs = self.after_norm(xs)
    # Here we assume the mask is not changed in encoder layers, so just
    # return the masks before encoder layers, and the masks will be used
    # for cross attention with decoder later
    return xs, masks

  def forward_chunk(
      self,
      xs: torch.Tensor,
      offset: int,
      required_cache_size: int,
      att_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      cnn_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      att_mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """ Forward just one chunk

        Args:
            xs (torch.Tensor): chunk input, with shape (b=1, time, mel-dim),
                where `time == (chunk_size - 1) * subsample_rate + \
                        subsample.right_context + 1`
            offset (int): current offset in encoder output time stamp
            required_cache_size (int): cache size required for next chunk
                compuation
                >=0: actual cache size
                <0: means all history cache is required
            att_cache (torch.Tensor): cache tensor for KEY & VALUE in
                transformer/conformer attention, with shape
                (elayers, head, cache_t1, d_k * 2), where
                `head * d_k == hidden-dim` and
                `cache_t1 == chunk_size * num_decoding_left_chunks`.
            cnn_cache (torch.Tensor): cache tensor for cnn_module in conformer,
                (elayers, b=1, hidden-dim, cache_t2), where
                `cache_t2 == cnn.lorder - 1`

        Returns:
            torch.Tensor: output of current input xs,
                with shape (b=1, chunk_size, hidden-dim).
            torch.Tensor: new attention cache required for next chunk, with
                dynamic shape (elayers, head, ?, d_k * 2)
                depending on required_cache_size.
            torch.Tensor: new conformer cnn cache required for next chunk, with
                same shape as the original cnn_cache.

        """
    assert xs.size(0) == 1
    # tmp_masks is just for interface compatibility
    tmp_masks = torch.ones(1,
                           xs.size(1),
                           device=xs.device,
                           dtype=torch.bool)
    tmp_masks = tmp_masks.unsqueeze(1)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    # NOTE(xcsong): Before embed, shape(xs) is (b=1, time, mel-dim)
    xs, pos_emb, _ = self.embed(xs, tmp_masks, offset)
    # NOTE(xcsong): After  embed, shape(xs) is (b=1, chunk_size, hidden-dim)
    elayers, cache_t1 = att_cache.size(0), att_cache.size(2)
    chunk_size = xs.size(1)
    attention_key_size = cache_t1 + chunk_size
    pos_emb = self.embed.position_encoding(
      offset=offset - cache_t1, size=attention_key_size)
    if required_cache_size < 0:
      next_cache_start = 0
    elif required_cache_size == 0:
      next_cache_start = attention_key_size
    else:
      next_cache_start = max(attention_key_size - required_cache_size, 0)
    r_att_cache = []
    r_cnn_cache = []
    for i, layer in enumerate(self.encoders):
      # NOTE(xcsong): Before layer.forward
      #   shape(att_cache[i:i + 1]) is (1, head, cache_t1, d_k * 2),
      #   shape(cnn_cache[i])       is (b=1, hidden-dim, cache_t2)
      xs, _, new_att_cache, new_cnn_cache = layer(
        xs, att_mask, pos_emb,
        att_cache=att_cache[i:i + 1] if elayers > 0 else att_cache,
        cnn_cache=cnn_cache[i] if cnn_cache.size(0) > 0 else cnn_cache
      )
      # NOTE(xcsong): After layer.forward
      #   shape(new_att_cache) is (1, head, attention_key_size, d_k * 2),
      #   shape(new_cnn_cache) is (b=1, hidden-dim, cache_t2)
      r_att_cache.append(new_att_cache[:, :, next_cache_start:, :])
      r_cnn_cache.append(new_cnn_cache.unsqueeze(0))
    if self.normalize_before:
      xs = self.after_norm(xs)

    # NOTE(xcsong): shape(r_att_cache) is (elayers, head, ?, d_k * 2),
    #   ? may be larger than cache_t1, it depends on required_cache_size
    r_att_cache = torch.cat(r_att_cache, dim=0)
    # NOTE(xcsong): shape(r_cnn_cache) is (e, b=1, hidden-dim, cache_t2)
    r_cnn_cache = torch.cat(r_cnn_cache, dim=0)

    return (xs, r_att_cache, r_cnn_cache)

  def forward_chunk_by_chunk(
      self,
      xs: torch.Tensor,
      decoding_chunk_size: int,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """ Forward input chunk by chunk with chunk_size like a streaming
            fashion

        Here we should pay special attention to computation cache in the
        streaming style forward chunk by chunk. Three things should be taken
        into account for computation in the current network:
            1. transformer/conformer encoder layers output cache
            2. convolution in conformer
            3. convolution in subsampling

        However, we don't implement subsampling cache for:
            1. We can control subsampling module to output the right result by
               overlapping input instead of cache left context, even though it
               wastes some computation, but subsampling only takes a very
               small fraction of computation in the whole model.
            2. Typically, there are several covolution layers with subsampling
               in subsampling module, it is tricky and complicated to do cache
               with different convolution layers with different subsampling
               rate.
            3. Currently, nn.Sequential is used to stack all the convolution
               layers in subsampling, we need to rewrite it to make it work
               with cache, which is not prefered.
        Args:
            xs (torch.Tensor): (1, max_len, dim)
            chunk_size (int): decoding chunk size
        """
    assert decoding_chunk_size > 0
    # The model is trained by static or dynamic chunk
    assert self.static_chunk_size > 0 or self.use_dynamic_chunk
    subsampling = self.embed.subsampling_rate
    context = self.embed.right_context + 1  # Add current frame
    stride = subsampling * decoding_chunk_size
    decoding_window = (decoding_chunk_size - 1) * subsampling + context
    num_frames = xs.size(1)
    att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    outputs = []
    offset = 0
    required_cache_size = decoding_chunk_size * num_decoding_left_chunks

    # Feed forward overlap input step by step
    for cur in range(0, num_frames - context + 1, stride):
      end = min(cur + decoding_window, num_frames)
      chunk_xs = xs[:, cur:end, :]
      (y, att_cache, cnn_cache) = self.forward_chunk(
        chunk_xs, offset, required_cache_size, att_cache, cnn_cache)
      outputs.append(y)
      offset += y.size(1)
    ys = torch.cat(outputs, 1)
    masks = torch.ones((1, 1, ys.size(1)), device=ys.device, dtype=torch.bool)
    return ys, masks


class AttentiveStatisticsPooling(torch.nn.Module):
  """This class implements an attentive statistic pooling layer for each channel.
  It returns the concatenated mean and std of the input tensor.

  Arguments
  ---------
  channels: int
      The number of input channels.
  output_channels: int
      The number of output channels.
  """

  def __init__(self, channels, output_channels):
    super().__init__()

    self._eps = 1e-12
    self._linear = Linear(channels * 3, channels)
    self._tanh = torch.nn.Tanh()
    self._conv = Conv1d(
      in_channels=channels, out_channels=channels, kernel_size=1
    )
    self._final_layer = torch.nn.Linear(channels * 2, output_channels,
                                        bias=False)
    logging.info("Init AttentiveStatisticsPooling with {}->{}".format(
      channels, output_channels))
    return

  @staticmethod
  def _compute_statistics(x: torch.Tensor,
                          m: torch.Tensor,
                          eps: float,
                          dim: int = 2):
    mean = (m * x).sum(dim)
    std = torch.sqrt((m * (x - mean.unsqueeze(dim)).pow(2)).sum(dim).clamp(eps))
    return mean, std

  def forward(self, x: torch.Tensor):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, L, C].
    """

    x = x.transpose(1, 2)
    L = x.shape[-1]
    lengths = torch.ones(x.shape[0], device=x.device)
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)
    total = mask.sum(dim=2, keepdim=True).float()

    mean, std = self._compute_statistics(x, mask / total, self._eps)
    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)
    attn = self._conv(self._tanh(self._linear(
      attn.transpose(1, 2)).transpose(1, 2)))

    attn = attn.masked_fill(mask == 0, float("-inf"))  # Filter out zero-padding
    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    pooled_stats = self._final_layer(torch.cat((mean, std), dim=1))
    return pooled_stats

  def forward_with_mask(self, x: torch.Tensor,
                        lengths: Optional[torch.Tensor] = None):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, C, L].
      lengths:
    """
    L = x.shape[-1]

    if lengths is None:
      lengths = torch.ones(x.shape[0], device=x.device)

    # Make binary mask of shape [N, 1, L]
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)

    # Expand the temporal context of the pooling layer by allowing the
    # self-attention to look at global properties of the utterance.

    # torch.std is unstable for backward computation
    # https://github.com/pytorch/pytorch/issues/4320
    total = mask.sum(dim=2, keepdim=True).float()
    mean, std = self._compute_statistics(x, mask / total, self._eps)

    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)

    # Apply layers
    attn = self.conv(self._tanh(self._linear(attn, lengths)))

    # Filter out zero-paddings
    attn = attn.masked_fill(mask == 0, float("-inf"))

    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    # Append mean and std of the batch
    pooled_stats = torch.cat((mean, std), dim=1)
    pooled_stats = pooled_stats.unsqueeze(2)
    return pooled_stats

  @staticmethod
  def length_to_mask(length: torch.Tensor,
                     max_len: Optional[int] = None,
                     dtype: Optional[torch.dtype] = None,
                     device: Optional[torch.device] = None):
    """Creates a binary mask for each sequence.

    Arguments
    ---------
    length : torch.LongTensor
        Containing the length of each sequence in the batch. Must be 1D.
    max_len : int
        Max length for the mask, also the size of the second dimension.
    dtype : torch.dtype, default: None
        The dtype of the generated mask.
    device: torch.device, default: None
        The device to put the mask variable.

    Returns
    -------
    mask : tensor
        The binary mask.

    Example
    -------
    """
    assert len(length.shape) == 1

    if max_len is None:
      max_len = length.max().long().item()  # using arange to generate mask
    mask = torch.arange(
      max_len, device=length.device, dtype=length.dtype
    ).expand(len(length), max_len) < length.unsqueeze(1)

    if dtype is None:
      dtype = length.dtype

    if device is None:
      device = length.device

    mask = torch.as_tensor(mask, dtype=dtype, device=device)
    return mask


class Model(BasicModel):
  """ Csi Backbone Model:
      Batch-norm
      Conformer-Encoder(x6 or x4)
      Global-Avg-Pool
      Resize and linear
      Bn-neck

  Loss:
    Focal-loss(ce) + Triplet-loss + Center-loss

  Args:
    hp: Dict with all parameters
  """

  def __init__(self, hp: Dict):
    super(BasicModel, self).__init__()
    self._hp = hp

    self._global_cmvn = torch.nn.BatchNorm1d(hp["input_dim"])
    self._encoder = ConformerEncoder(
      input_size=hp["input_dim"],
      output_size=hp["encoder"]["output_dims"],
      linear_units=hp["encoder"]["attention_dim"],
      num_blocks=hp["encoder"]["num_blocks"])

    if hp["encoder"]["output_dims"] != hp["embed_dim"]:
      self._embed_lo = torch.nn.Linear(hp["encoder"]["output_dims"],
                                       hp["embed_dim"])
    else:
      self._embed_lo = None

    # Bottleneck
    self._bottleneck = torch.nn.BatchNorm1d(hp["embed_dim"])
    self._bottleneck.bias.requires_grad_(False)  # no shift

    self._pool_layer = AttentiveStatisticsPooling(
      hp["embed_dim"], output_channels=hp["embed_dim"])
    self._ce_layer = torch.nn.Linear(
      hp["embed_dim"], hp["ce"]["output_dims"], bias=False)

    # Loss
    if "alpha" in hp["ce"].keys():
      alpha = np.load(hp["ce"]["alpha"])
      alpha = 1.0 / (alpha + 1)
      alpha = alpha / np.sum(alpha)
      logging.warning("use alpha with {}".format(len(alpha)))
    else:
      alpha = None
      logging.warning("Not use alpha")

    self._ce_loss = FocalLoss(alpha=alpha, gamma=self._hp["ce"]["gamma"],
                              num_cls=self._hp["ce"]["output_dims"])
    self._triplet_loss = HardTripletLoss(margin=hp["triplet"]["margin"])
    self._center_loss = CenterLoss(num_classes=self._hp["ce"]["output_dims"],
                                   feat_dim=hp["embed_dim"], use_gpu=torch.cuda.is_available())
    logging.info(
      "Model size: {:.3f}M \n".format(self.model_size() / 1000 / 1000))
    return

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """feat[b, frame_size, feat_size] -> embed[b, embed_dim]"""

    x = self._global_cmvn(x.transpose(1, 2)).transpose(1, 2)
    xs_lens = torch.full(
      [x.size(0)], fill_value=x.size(1), dtype=torch.long).to(x.device).long()
    x, _ = self._encoder(x, xs_lens=xs_lens, decoding_chunk_size=-1)
    x = self._pool_layer(x)
    return x

  @torch.jit.ignore
  def compute_loss(self, anchor: torch.Tensor, label: torch.Tensor) -> Tuple[
    torch.Tensor, Dict]:
    """compute ce and triplet loss"""
    f_t = self.forward(anchor)
    # print("f_t", f_t)

    f_i = self._bottleneck(f_t)
    ce_pred = self._ce_layer(f_i)
    # print("ce_pred", ce_pred)
    ce_loss = self._ce_loss(ce_pred, label)
    loss = ce_loss * self._hp["ce"]["weight"]
    loss_dict = {"ce_loss": ce_loss}

    tri_weight = self._hp["triplet"]["weight"]
    if tri_weight > 0.01:
      tri_loss = self._triplet_loss(f_t, label)
      loss = loss + tri_loss * tri_weight
      loss_dict.update({"tri_loss": tri_loss})

    cen_weight = self._hp["center"]["weight"]
    if cen_weight > 0.0:
      cen_loss = self._center_loss(f_t, label)
      loss = loss + cen_loss * cen_weight
      loss_dict.update({"cen_loss": cen_loss})
    return loss, loss_dict

  @torch.jit.ignore
  def inference(self, feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    with torch.no_grad():
      embed = self.forward(feat)
      embed_ce = self._ce_layer(embed)
    return embed, embed_ce

  @torch.jit.export
  def compute_embed(self, feat: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
      embed = self.forward(feat)
    return embed


### 2.1 Lop tien xu ly tin hieu am thanh (AudioPreprocessor)
Resample am thanh ve 16,000 Hz, chuan hoa bien do cuc dai ve [-1.0, 1.0], va cat thanh cac cua so truot 10.0 giay voi buoc nhay 3.0 giay.

In [ ]:
class AudioPreprocessor:
    """Tiền xử lý âm thanh: Resample về 16,000 Hz, chuẩn hóa biên độ và cắt Sliding Window 10s / hop 3s."""
    def __init__(self, sample_rate: int = 16000, chunk_duration: float = 10.0, hop_duration: float = 3.0):
        self.sample_rate = sample_rate
        self.chunk_duration = chunk_duration
        self.hop_duration = hop_duration
        self.chunk_samples = int(chunk_duration * sample_rate)
        self.hop_samples = int(hop_duration * sample_rate)

    def load_audio(self, audio_path: str) -> np.ndarray:
        """Nạp và chuẩn hóa âm thanh, không nuốt lỗi âm thầm."""
        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Không tìm thấy file âm thanh: {audio_path}")
            
        if audio_path.lower().endswith('.silent'):
            return np.zeros(self.chunk_samples, dtype=np.float32)

        try:
            if audio_path.lower().endswith('.mp3'):
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            else:
                y, sr = sf.read(audio_path, always_2d=False, dtype='float32')
                if len(y.shape) > 1:
                    y = np.mean(y, axis=1)
                if sr != self.sample_rate:
                    y = librosa.resample(y, orig_sr=sr, target_sr=self.sample_rate)
        except Exception as e:
            # Fallback thu doc bang librosa neu soundfile gap loi dinh dang wav
            try:
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            except Exception as final_e:
                print(f"[!] Canh bao: Khong the doc file {audio_path}: {final_e}")
                raise RuntimeError(f"Không thể đọc file âm thanh: {audio_path}") from final_e

        y = np.asarray(y, dtype=np.float32).reshape(-1)
        if y.size == 0:
            raise ValueError(f"File audio rong: {audio_path}")
        if not np.isfinite(y).all():
            raise ValueError(f"Audio chua NaN/Inf: {audio_path}")
        max_val = float(np.max(np.abs(y)))
        if max_val > 1e-6:
            y = y / max_val
        return y

    def slice_into_chunks(self, y: np.ndarray) -> list:
        """Cắt âm thanh thành danh sách các đoạn 10 giây (hop 3s). Trả về ít nhất 1 đoạn."""
        if len(y) <= self.chunk_samples:
            return [np.pad(y, (0, self.chunk_samples - len(y)), mode='constant')]
        chunks = []
        start = 0
        for start in range(0, len(y) - self.chunk_samples + 1, self.hop_samples):
            chunks.append(y[start : start + self.chunk_samples])
        last_chunk_end = start + self.chunk_samples
        remaining_unseen = len(y) - last_chunk_end
        if remaining_unseen > (self.chunk_samples // 3):
            final_chunk = y[-self.chunk_samples:]
            if not np.array_equal(final_chunk, chunks[-1]):
                chunks.append(final_chunk)
        return chunks if chunks else [np.pad(y, (0, max(0, self.chunk_samples - len(y))), mode='constant')[:self.chunk_samples]]

### 2.2 Lop trich xuat dac trung pho Constant-Q Transform (CoverHunterCQT)
Trich xuat pho CQT 96 bins tan so (12 bins/octave, 8 quang tam tu 32 Hz den 8000 Hz), hop 0.04s, nen thoi gian qua ham shorter(mean_size=3) va chuan hoa dB theo dung chuan CoverHunter goc.

In [ ]:
import scipy.sparse

def shorter(feat: np.ndarray, mean_size: int = 3) -> np.ndarray:
    """Nén trục thời gian của ma trận đặc trưng CQT theo bước mean_size (CoverHunter chuẩn)."""
    if mean_size <= 1:
        return feat
    cqt = feat.T
    height, length = cqt.shape
    new_len = int(length / mean_size)
    new_cqt = np.zeros((height, new_len), dtype=np.float32)
    for i in range(new_len):
        new_cqt[:, i] = cqt[:, i * mean_size : (i + 1) * mean_size].mean(axis=1)
    return new_cqt.T

class PyCqt:
    """Thuật toán CQT Sparse Kernel chính thức của CoverHunter (Tsinghua/MIREX)."""
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = None):
        self._hop_size = hop_size
        self._sample_rate = sample_rate
        if not max_freq:
            max_freq = sample_rate // 2
        self._kernel = self._compute_cqt_kernel(sample_rate, octave_resolution, min_freq, max_freq)

    @staticmethod
    def _compute_cqt_kernel(sampling_frequency: int, octave_resolution: int, minimum_frequency: float, maximum_frequency: float):
        quality_factor = 1.0 / (pow(2.0, 1.0 / octave_resolution) - 1.0)
        number_frequencies = round(octave_resolution * np.log2(maximum_frequency / minimum_frequency))
        fft_length = int(pow(2, np.ceil(np.log2(quality_factor * sampling_frequency / minimum_frequency))))
        cqt_kernel = np.zeros((number_frequencies, fft_length), dtype=complex)

        for i in range(number_frequencies):
            frequency_value = minimum_frequency * pow(2.0, i / octave_resolution)
            window_length = 2 * round(quality_factor * sampling_frequency / frequency_value / 2.0) + 1
            temporal_kernel = (np.hamming(window_length)
                               * np.exp(2.0 * np.pi * 1j * quality_factor
                                        * np.arange(-(window_length - 1) / 2.0, (window_length - 1) / 2.0 + 1) / window_length)
                               / window_length)
            pad_width = int((fft_length - window_length + 1) / 2)
            cqt_kernel[i, pad_width : pad_width + window_length] = temporal_kernel

        cqt_kernel = np.fft.fft(cqt_kernel, axis=1)
        cqt_kernel[np.absolute(cqt_kernel) < 0.01] = 0
        cqt_kernel = scipy.sparse.csr_matrix(cqt_kernel)
        cqt_kernel = np.conjugate(cqt_kernel) / fft_length
        return cqt_kernel

    @staticmethod
    def _compute_cqt_spec(audio_signal: np.ndarray, sampling_frequency: int, time_resolution: int, cqt_kernel):
        step_length = round(sampling_frequency / time_resolution)
        number_times = int(np.floor(len(audio_signal) / step_length))
        number_frequencies, fft_length = np.shape(cqt_kernel)

        pad_left = int(np.ceil((fft_length - step_length) / 2.0))
        pad_right = int(np.floor((fft_length - step_length) / 2.0))
        audio_signal = np.pad(audio_signal, (pad_left, pad_right), "constant", constant_values=(0, 0))

        cqt_spectrogram = np.zeros((number_frequencies, number_times), dtype=np.float32)
        idx = 0
        for j in range(number_times):
            cqt_spectrogram[:, j] = np.absolute(cqt_kernel * np.fft.fft(audio_signal[idx : idx + fft_length]))
            idx += step_length
        return cqt_spectrogram

    def compute_cqt(self, signal_float: np.ndarray, feat_dim_first: bool = False) -> np.ndarray:
        time_resolution = int(round(1.0 / self._hop_size))
        cqt_spectrogram = self._compute_cqt_spec(signal_float, self._sample_rate, time_resolution, self._kernel)
        magnitude = np.maximum(cqt_spectrogram, 1e-9)
        ref_value = max(float(np.max(magnitude)), 1e-4)
        cqt_spectrogram = 20.0 * np.log10(magnitude) - 20.0 * np.log10(ref_value)
        cqt_spectrogram = np.clip(cqt_spectrogram, -80.0, 0.0)
        if not feat_dim_first:
            cqt_spectrogram = cqt_spectrogram.T
        return cqt_spectrogram

class CoverHunterCQT:
    """
    Trích xuất đặc trưng CQT chuẩn CoverHunter:
    - 96 bins (12 bins/octave, 8 octaves từ 32 Hz đến 8000 Hz)
    - hop_size: 0.04s (25 fps), nén thời gian shorter(mean_size=3)
    - Chuẩn hóa dB theo biên độ cực đại: 20 * log10(cqt / max_cqt)
    """
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = 8000.0, mean_size: int = 3):
        self.sample_rate = sample_rate
        self.mean_size = mean_size
        self.py_cqt = PyCqt(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=octave_resolution,
            min_freq=min_freq,
            max_freq=max_freq
        )

    def extract(self, audio: np.ndarray) -> torch.Tensor:
        if isinstance(audio, torch.Tensor):
            audio = audio.detach().cpu().numpy()
        feat = self.py_cqt.compute_cqt(audio, feat_dim_first=False) # (T, 96)
        if self.mean_size > 1:
            feat = shorter(feat, mean_size=self.mean_size)
        return torch.from_numpy(feat).float()

### 2.3 Lop can chinh kich thuoc pho thoi gian (CQTExtractor)
Ho tro 2 che do trich xuat: do dai co dinh 83 frames (10s) phuc vu training batch va FAISS global embedding, hoac do dai dong (variable_length=True) giu nguyen so frames thuc te phuc vu can chinh DTW.

In [ ]:
class CQTExtractor:
    """Trích xuất Constant-Q Transform (CQT) hỗ trợ cả chế độ Fixed-Length (83 frames) và Variable-Length."""
    def __init__(self, sample_rate: int = 16000, n_bins: int = 96, bins_per_octave: int = 12, hop_size: float = 0.04, mean_size: int = 3, target_frames: int = 83):
        self.sample_rate = sample_rate
        self.n_bins = n_bins
        self.bins_per_octave = bins_per_octave
        self.hop_size = hop_size
        self.mean_size = mean_size
        self.target_frames = target_frames
        self.cqt_impl = CoverHunterCQT(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=bins_per_octave,
            min_freq=32.0,
            max_freq=8000.0,
            mean_size=mean_size
        )

    def extract(self, y: np.ndarray, variable_length: bool = False) -> torch.Tensor:
        """
        Trích xuất tensor CQT từ tín hiệu âm thanh 1D.
        - variable_length=False: Căn chỉnh về đúng [1, 96, target_frames] (mặc định 83 frames ~ 10s).
        - variable_length=True: Giữ nguyên độ dài thời gian thực tế [1, 96, T] phục vụ Stage 2 DTW.
        """
        feat = self.cqt_impl.extract(y)  # (T, 96)
        tensor = feat.T.unsqueeze(0)      # (1, 96, T)

        if not variable_length and self.target_frames is not None and self.target_frames > 0:
            curr_t = tensor.shape[-1]
            min_val = tensor.min()
            if curr_t < self.target_frames:
                tensor = F.pad(tensor, (0, self.target_frames - curr_t), mode='constant', value=min_val.item())
            elif curr_t > self.target_frames:
                tensor = tensor[:, :, :self.target_frames]
        return tensor

SAMPLE_RATE = 16000
CHUNK_DURATION_SEC = 10.0
CHUNK_HOP_DURATION_SEC = 3.0

prep = AudioPreprocessor(sample_rate=SAMPLE_RATE, chunk_duration=CHUNK_DURATION_SEC, hop_duration=CHUNK_HOP_DURATION_SEC)
cqt_ext = CQTExtractor(sample_rate=16000, n_bins=96, bins_per_octave=12, hop_size=0.04, mean_size=3, target_frames=83)
print("[+] Khởi tạo thành công AudioPreprocessor và CoverHunter CQT Standalone (96 Bins, 83 Frames, dB Scale)!")

### 3.1 Ham dich cao do bat bien quang tam (apply_cqt_pitch_shift)
Dich chuyen truc tan so cua CQT de mo phong nguoi hat lech tone. Khac phuc loi am hoc: dien cac bin trong bang muc san nhieu `cqt.min()` (khoang lang -140 dB), tuyet doi khong dien 0.0.

In [ ]:
def apply_cqt_pitch_shift(cqt: torch.Tensor, shift_bins: int) -> torch.Tensor:
    """
    Dịch chuyển cao độ trên ma trận CQT bằng cách tịnh tiến trục tần số (dim -2).
    - shift_bins > 0: Dịch lên cao độ cao hơn (tăng semitone).
    - shift_bins < 0: Dịch xuống cao độ thấp hơn (giảm semitone).
    - Điền các bin trống bằng cqt.min() (mức sàn im lặng), KHÔNG điền 0.0 (vốn là 0 dB cực đại).
    """
    if shift_bins == 0:
        return cqt

    min_val = cqt.min()
    shifted = torch.full_like(cqt, fill_value=min_val)
    num_bins = cqt.shape[-2]

    if shift_bins > 0:
        if shift_bins < num_bins:
            shifted[..., shift_bins:, :] = cqt[..., :num_bins - shift_bins, :]
    else:
        abs_shift = abs(shift_bins)
        if abs_shift < num_bins:
            shifted[..., :num_bins - abs_shift, :] = cqt[..., abs_shift:, :]

    return shifted

### 3.2 Lop tang cuong dich tone ngau nhien (CQTPitchShiftAugmentation)
Dich ngau nhien trong dai [-12, +12] semitones tron 1 quang 8 (xac suat 80%), giup mo hinh bat bien hoan toan truoc su khac biet cao do giua giong Nam va giong Nu.

In [ ]:
class CQTPitchShiftAugmentation:
    """
    Tăng cường dữ liệu dịch tone ngẫu nhiên (Full-Octave Pitch-Shift Data Augmentation).
    Dịch ngẫu nhiên trong dải [-max_shift_bins, +max_shift_bins] nửa cung (semitones).
    Mặc định dải ±12 semitones bao trọn 1 quãng 8 (chuyển đổi hoàn hảo giữa giọng Nam và Nữ).
    """
    def __init__(self, max_shift_bins: int = 12, prob: float = 0.8):
        self.max_shift_bins = max_shift_bins
        self.prob = prob

    def __call__(self, cqt_tensor: torch.Tensor) -> torch.Tensor:
        if random.random() > self.prob:
            return cqt_tensor
        shift = random.randint(-self.max_shift_bins, self.max_shift_bins)
        return apply_cqt_pitch_shift(cqt_tensor, shift)

### 3.3 Lop bien dang toc do va che phu pho (SpectrogramTempoAndMaskDistortion)
Co gian truc thoi gian trong khoang [0.8, 1.2] mo phong hat nhanh/cham ma giu nguyen cao do, ket hop SpecAugment che phu nhe mo phong ngat tieng hoac micro re.

In [ ]:
class SpectrogramTempoAndMaskDistortion:
    """Nonlinear time warp plus SpecAugment masks; the CQT frequency axis is never warped."""
    def __init__(self, time_scale_range=(0.5, 1.5), max_mask_t=0, max_mask_f=0, prob=0.6, knots=6):
        self.scale_range = time_scale_range
        self.max_mask_t = int(max_mask_t)
        self.max_mask_f = int(max_mask_f)
        self.prob = prob
        self.knots = knots
        if not 0 < time_scale_range[0] <= time_scale_range[1] or knots < 2:
            raise ValueError('Invalid distortion parameters')
        if self.max_mask_t < 0 or self.max_mask_f < 0:
            raise ValueError('Mask widths must be non-negative')

    def _axis(self, size, batch, device, dtype):
        increments = torch.empty(batch, 1, self.knots, device=device, dtype=dtype).uniform_(*self.scale_range)
        positions = torch.cat([torch.zeros(batch, 1, 1, device=device, dtype=dtype), increments.cumsum(-1)], -1)
        positions = 2 * positions / positions[..., -1:] - 1
        return F.interpolate(positions, size=size, mode='linear', align_corners=True)[:, 0]

    @staticmethod
    def _mask_axis(x, axis, max_width):
        if max_width <= 0 or x.shape[axis] <= 1:
            return
        width = random.randint(1, min(max_width, x.shape[axis] - 1))
        start = random.randint(0, x.shape[axis] - width)
        slices = [slice(None)] * x.ndim
        slices[axis] = slice(start, start + width)
        fill_value = x.amin(dim=(-2, -1), keepdim=True)
        x[tuple(slices)] = fill_value

    def __call__(self, spec):
        if random.random() >= self.prob:
            return spec
        was_unbatched = spec.ndim == 3
        x = spec.unsqueeze(0) if was_unbatched else spec
        b, _, h, w = x.shape
        tx = self._axis(w, b, x.device, x.dtype)
        fy = torch.linspace(-1, 1, h, device=x.device, dtype=x.dtype).expand(b, h)
        grid = torch.stack([tx[:, None, :].expand(b, h, w), fy[:, :, None].expand(b, h, w)], -1)
        out = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True).clone()
        self._mask_axis(out, axis=-1, max_width=self.max_mask_t)
        self._mask_axis(out, axis=-2, max_width=self.max_mask_f)
        return out.squeeze(0) if was_unbatched else out


### Dataset and bounded CQT cache
CQT is cached on disk; the RAM LRU is limited by LOCAL_RAM_CACHE_MB.


In [ ]:
from collections import OrderedDict
import hashlib

class BoundedCQTCache(OrderedDict):
    def __init__(self, max_mb):
        super().__init__()
        self.limit = int(max_mb * 1024**2)
        self.bytes_used = 0

    @staticmethod
    def size(value):
        return sum(t.numel() * t.element_size() for t in value)

    def __getitem__(self, key):
        value = super().__getitem__(key)
        self.move_to_end(key)
        return value

    def __setitem__(self, key, value):
        if key in self:
            self.bytes_used -= self.size(super().__getitem__(key))
            super().__delitem__(key)
        size = self.size(value)
        if size > self.limit:
            return
        while self and self.bytes_used + size > self.limit:
            _, old = self.popitem(last=False)
            self.bytes_used -= self.size(old)
        super().__setitem__(key, value)
        self.bytes_used += size

GLOBAL_CQT_RAM_CACHE = BoundedCQTCache(LOCAL_RAM_CACHE_MB)
CQT_CACHE_DIR = STAGE2_ROOT / 'cache' / 'cqt_timegrid_v4_16k_10s_hop3_96x83'
CQT_CACHE_DIR.mkdir(parents=True, exist_ok=True)


class ByteHumCachedTripletDataset(Dataset):
    """
    Tập dữ liệu sinh bộ ba Triplet (Anchor, Positive, Negative) với cơ chế In-RAM CQT Caching:
    - Lưu danh sách tất cả các chunk CQT của mỗi file audio vào RAM, khắc phục lỗi chỉ lấy chunks[0].
    - Khi training: ngẫu nhiên chọn 1 chunk từ danh sách chunks cho mỗi epoch để học đa dạng vị trí.
    - Khi validation: lấy cố định chunks[0] để kết quả đánh giá mang tính xác định (deterministic).
    - Áp dụng Pitch-Shift (±12 semitones) và Tempo Distortion ngẫu nhiên động trên CQT.
    """
    def __init__(
        self,
        split_dir: str,
        audio_preprocessor: AudioPreprocessor,
        cqt_extractor: CQTExtractor,
        tempo_distortion: Optional[SpectrogramTempoAndMaskDistortion] = None,
        pitch_shifter: Optional[CQTPitchShiftAugmentation] = None,
        is_training: bool = True,
        song_to_idx: Optional[Dict[str, int]] = None,
        vocal_ratio: float = 0.80
    ):
        self.split_dir = split_dir
        self.audio_preprocessor = audio_preprocessor
        self.cqt_extractor = cqt_extractor
        self.tempo_distortion = tempo_distortion
        self.pitch_shifter = pitch_shifter
        self.is_training = is_training
        self.vocal_ratio = vocal_ratio

        if not split_dir or not os.path.exists(split_dir):
            raise FileNotFoundError(f"Không tìm thấy thư mục: {split_dir}")

        self.songs = sorted([s for s in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, s))])
        if song_to_idx is not None:
            self.song_to_idx = song_to_idx
        else:
            self.song_to_idx = {s: i for i, s in enumerate(self.songs)}

        self.song_data = {}
        self.humming_samples = []

        for s in self.songs:
            s_path = os.path.join(split_dir, s)
            frags = sorted([f for f in os.listdir(s_path) if os.path.isdir(os.path.join(s_path, f))])
            self.song_data[s] = {}
            for f in frags:
                f_path = os.path.join(s_path, f)
                audio_extensions = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.silent')
                audio_files = sorted(w for w in os.listdir(f_path) if w.lower().endswith(audio_extensions))
                refs = [os.path.join(f_path, w) for w in audio_files if w == 'reference.wav']
                vocals = [os.path.join(f_path, w) for w in audio_files if w.startswith('reference_vocal')]
                hums = [os.path.join(f_path, w) for w in audio_files if not w.startswith('reference')]
                if refs or vocals or hums:
                    self.song_data[s][f] = {'refs': refs, 'vocals': vocals, 'hums': hums}
                    for h in hums:
                        self.humming_samples.append((s, f, h))

    def __len__(self) -> int:
        return len(self.humming_samples)

    def _get_raw_cqts(self, path: str) -> List[torch.Tensor]:
        """Lấy danh sách tất cả các tensor CQT chunks từ RAM Cache, nếu chưa có thì trích xuất toàn bộ."""
        if path in GLOBAL_CQT_RAM_CACHE:
            return GLOBAL_CQT_RAM_CACHE[path]
        source = Path(path).resolve()
        stat = source.stat()
        fingerprint = f'{source}|{stat.st_size}|{stat.st_mtime_ns}|v4_16k_10s_hop3_96x83'
        cache_path = CQT_CACHE_DIR / (hashlib.sha256(fingerprint.encode()).hexdigest() + '.pt')
        if cache_path.is_file():
            cqts = torch.load(cache_path, map_location='cpu', weights_only=True)
            if not cqts or any(t.shape != (1, 96, 83) or not torch.isfinite(t).all() for t in cqts):
                raise RuntimeError(f'Invalid CQT cache; remove this file: {cache_path}')
            GLOBAL_CQT_RAM_CACHE[path] = cqts
            return cqts
        try:
            y = self.audio_preprocessor.load_audio(path)
            chunks = self.audio_preprocessor.slice_into_chunks(y)
            cqts = [self.cqt_extractor.extract(chk, variable_length=False) for chk in chunks]
        except Exception as e:
            raise RuntimeError(f"Không thể tạo CQT cho {path}") from e
        temp_path = cache_path.with_suffix('.tmp')
        torch.save(cqts, temp_path)
        os.replace(temp_path, cache_path)
        GLOBAL_CQT_RAM_CACHE[path] = cqts
        return cqts

    def _sample_chunk(self, path: str) -> torch.Tensor:
        """Chọn 1 chunk CQT: ngẫu nhiên khi train, cố định chunk 0 khi val."""
        cqts = self._get_raw_cqts(path)
        if self.is_training:
            return random.choice(cqts).clone()
        return cqts[0].clone()

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        song_a, frag_a, hum_a = self.humming_samples[idx]
        
        # 1. Anchor: Tiếng ngân nga (Domain 1.0)
        cqt_a = self._sample_chunk(hum_a)
        if self.is_training:
            if self.tempo_distortion: cqt_a = self.tempo_distortion(cqt_a)
            if self.pitch_shifter: cqt_a = self.pitch_shifter(cqt_a)

        # 2. Positive: Bản thu gốc hoặc vocal tách rời
        pool = self.song_data[song_a][frag_a]
        if self.is_training and pool['vocals'] and random.random() < self.vocal_ratio:
            pos_path, domain_p = random.choice(pool['vocals']), 0.0
        elif pool['refs']:
            pos_path, domain_p = (random.choice(pool['refs']) if self.is_training else pool['refs'][0]), 0.0
        elif pool['vocals']:
            pos_path, domain_p = (random.choice(pool['vocals']) if self.is_training else pool['vocals'][0]), 0.0
        else:
            cands = sorted(h for h in pool['hums'] if h != hum_a)
            if not cands:
                raise RuntimeError(f"Thiếu positive phù hợp cho {song_a}/{frag_a}")
            pos_path, domain_p = (random.choice(cands) if self.is_training else cands[0]), 1.0
            
        cqt_p = self._sample_chunk(pos_path)
        if self.is_training and domain_p == 1.0:
            if self.pitch_shifter: cqt_p = self.pitch_shifter(cqt_p)

        # 3. Negative: Bản thu của bài hát khác hoàn toàn
        neg_song_candidates = [s for s in self.songs if s != song_a and self.song_data[s]]
        if not neg_song_candidates:
            raise RuntimeError(f"Không có negative hợp lệ cho {song_a}")
        if self.is_training:
            neg_song = random.choice(neg_song_candidates)
            neg_frag = random.choice(sorted(self.song_data[neg_song]))
        else:
            neg_song = neg_song_candidates[idx % len(neg_song_candidates)]
            neg_fragments = sorted(self.song_data[neg_song])
            neg_frag = neg_fragments[idx % len(neg_fragments)]
        neg_pool = self.song_data[neg_song][neg_frag]
        if self.is_training:
            if random.random() > 0.5 and neg_pool['refs']:
                neg_path, domain_n = random.choice(neg_pool['refs']), 0.0
            elif neg_pool['hums']:
                neg_path, domain_n = random.choice(neg_pool['hums']), 1.0
            elif neg_pool['vocals']:
                neg_path, domain_n = random.choice(neg_pool['vocals']), 0.0
            else:
                raise RuntimeError(f"Thiếu negative audio cho {neg_song}/{neg_frag}")
        elif neg_pool['refs']:
            neg_path, domain_n = neg_pool['refs'][0], 0.0
        elif neg_pool['hums']:
            neg_path, domain_n = neg_pool['hums'][0], 1.0
        elif neg_pool['vocals']:
            neg_path, domain_n = neg_pool['vocals'][0], 0.0
        else:
            raise RuntimeError(f"Thiếu negative audio cho {neg_song}/{neg_frag}")
            
        cqt_n = self._sample_chunk(neg_path)

        song_label = self.song_to_idx[song_a]
        neg_song_label = self.song_to_idx[neg_song]

        return {
            'anchor': cqt_a,
            'positive': cqt_p,
            'negative': cqt_n,
            'song_idx': torch.tensor(song_label, dtype=torch.long),
            'negative_song_idx': torch.tensor(neg_song_label, dtype=torch.long),
            'domain_anchor': torch.tensor([1.0], dtype=torch.float),
            'domain_positive': torch.tensor([domain_p], dtype=torch.float),
            'domain_negative': torch.tensor([domain_n], dtype=torch.float)
        }

### Song-balanced local DataLoaders
Default microbatch: 2 songs x 2 queries; Windows uses num_workers=0.


In [ ]:
aug_tempo_train = SpectrogramTempoAndMaskDistortion(time_scale_range=(0.7, 1.3), max_mask_t=8, max_mask_f=4, prob=0.6)
aug_pitch_train = CQTPitchShiftAugmentation(max_shift_bins=12, prob=0.8)

train_dataset = ByteHumCachedTripletDataset(
    split_dir=TRAIN_DIR,
    audio_preprocessor=prep,
    cqt_extractor=cqt_ext,
    tempo_distortion=aug_tempo_train,
    pitch_shifter=aug_pitch_train,
    is_training=True,
    vocal_ratio=LOCAL_POSITIVE_VOCAL_RATIO
)

val_dataset = ByteHumCachedTripletDataset(
    split_dir=VAL_DIR,
    audio_preprocessor=prep,
    cqt_extractor=cqt_ext,
    tempo_distortion=None,
    pitch_shifter=None,
    is_training=False,
    vocal_ratio=0.80
)


def prewarm_cqt_cache(datasets, max_workers=None):
    """Precompute every unique audio file once so epoch 1 is not I/O bound."""
    from concurrent.futures import ThreadPoolExecutor

    paths = set()
    for dataset in datasets:
        for fragments in dataset.song_data.values():
            for pool in fragments.values():
                paths.update(pool['refs'])
                paths.update(pool['vocals'])
                paths.update(pool['hums'])
    pending = sorted(path for path in paths if path not in GLOBAL_CQT_RAM_CACHE)
    workers = max_workers or min(8, os.cpu_count() or 1)
    if not pending:
        return
    print(f"[*] Pre-warm CQT cache: {len(pending):,} files voi {workers} workers...")
    owner = datasets[0]
    with ThreadPoolExecutor(max_workers=workers) as executor:
        list(tqdm(executor.map(owner._get_raw_cqts, pending), total=len(pending), desc='CQT cache', leave=False))
    print(f"[+] CQT cache san sang: {len(GLOBAL_CQT_RAM_CACHE):,} files")


PREWARM_CQT_CACHE = os.environ.get('QBH_PREWARM_CQT', '0') == '1'
# Kiem tra chong ro ri du lieu truoc cac cong viec cache ton CPU/RAM.
test_song_ids = {s for s in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, s))} if TEST_DIR and os.path.exists(TEST_DIR) else set()
split_song_ids = {'train': set(train_dataset.songs), 'val': set(val_dataset.songs), 'test': test_song_ids}
for left, right in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = split_song_ids[left] & split_song_ids[right]
    assert not overlap, f"Phat hien ro ri du lieu {left}/{right}: {sorted(overlap)}"
print(f"[+] Kiem tra leakage PASS: train={len(split_song_ids['train'])}, val={len(split_song_ids['val'])}, test={len(split_song_ids['test'])} bai; moi giao tap = 0.")

if PREWARM_CQT_CACHE:
    prewarm_cqt_cache([train_dataset, val_dataset])

class SongBalancedBatchSampler:
    """Tao moi batch theo cau truc P bai hat x K query de tang negative hop le."""
    def __init__(self, dataset, songs_per_batch=8, samples_per_song=2):
        self.dataset = dataset
        self.songs_per_batch = songs_per_batch
        self.samples_per_song = samples_per_song
        self.batch_size = songs_per_batch * samples_per_song
        self.song_to_indices = {}
        for index, (song, _, _) in enumerate(dataset.humming_samples):
            self.song_to_indices.setdefault(song, []).append(index)
        self.songs = sorted(self.song_to_indices)
        if len(self.songs) < songs_per_batch:
            raise ValueError(f"Can it nhat {songs_per_batch} bai train, chi tim thay {len(self.songs)}")
        self.num_batches = max(1, len(dataset) // self.batch_size)

    def __len__(self):
        return self.num_batches

    def __iter__(self):
        song_order = self.songs.copy()
        random.shuffle(song_order)
        cursor = 0
        for _ in range(self.num_batches):
            if cursor + self.songs_per_batch > len(song_order):
                random.shuffle(song_order)
                cursor = 0
            selected_songs = song_order[cursor:cursor + self.songs_per_batch]
            cursor += self.songs_per_batch
            batch = []
            for song in selected_songs:
                indices = self.song_to_indices[song]
                if len(indices) >= self.samples_per_song:
                    batch.extend(random.sample(indices, self.samples_per_song))
                else:
                    batch.extend(random.choices(indices, k=self.samples_per_song))
            random.shuffle(batch)
            yield batch


SONGS_PER_BATCH = LOCAL_SONGS_PER_BATCH
SAMPLES_PER_SONG = LOCAL_SAMPLES_PER_SONG
BATCH_SIZE = SONGS_PER_BATCH * SAMPLES_PER_SONG
NUM_WORKERS = 0

def seed_worker(worker_id):
    worker_seed = (torch.initial_seed() + worker_id) % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(42)

train_batch_sampler = SongBalancedBatchSampler(
    train_dataset,
    songs_per_batch=SONGS_PER_BATCH,
    samples_per_song=SAMPLES_PER_SONG
)
train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=loader_generator
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=loader_generator
)

print(f"[+] Khoi tao Dataset va Song-Balanced Batch Sampler:")
print(f"    - Mau Triplet Train : {len(train_dataset):,} | Mau Triplet Val: {len(val_dataset):,}")
print(f"    - Batch             : {SONGS_PER_BATCH} bai x {SAMPLES_PER_SONG} query = {BATCH_SIZE}")
print(f"    - Train Batches     : {len(train_loader):,} | Workers: {NUM_WORKERS}")

### 4.3 Kiem tra kich thuoc tensor cua mot batch du lieu
Lay thu nghiem 1 batch tu train_loader de kiem tra kich thuoc cua Anchor, Positive, Negative cung cac nhan dinh danh bai hat va mien du lieu.

In [ ]:
sample_batch = next(iter(train_loader))

assert sample_batch['anchor'].shape == (BATCH_SIZE, 1, 96, 83), f"Anchor shape sai: {sample_batch['anchor'].shape}"
assert sample_batch['positive'].shape == (BATCH_SIZE, 1, 96, 83), f"Positive shape sai: {sample_batch['positive'].shape}"
assert sample_batch['negative'].shape == (BATCH_SIZE, 1, 96, 83), f"Negative shape sai: {sample_batch['negative'].shape}"
assert sample_batch['song_idx'].shape == (BATCH_SIZE,), f"Song idx shape sai: {sample_batch['song_idx'].shape}"

print("[+] Kiem tra kich thuoc 1 batch du lieu tu train_loader (PASS):")
print(f"    - Anchor CQT shape   : {sample_batch['anchor'].shape} (Chuan [B, 1, 96, 83])")
print(f"    - Positive CQT shape : {sample_batch['positive'].shape}")
print(f"    - Negative CQT shape : {sample_batch['negative'].shape}")
print(f"    - Song IDs mau       : {sample_batch['song_idx'][:4].tolist()}")
print(f"    - Domain Targets mau : {sample_batch['domain_anchor'][:4].squeeze().tolist()}")
print(f"    - So file CQT da luu trong RAM cache: {len(GLOBAL_CQT_RAM_CACHE)}")

### 5.1 Dinh nghia tang dao nguoc dao ham (Gradient Reversal Layer - GRL)
Dao nguoc chieu gradient (nhan voi -lambda) khi truyen nguoc tu bo phan loai mien sang feature extractor, giup mo hinh hoc dac trung bat bien mien am thanh (Domain Invariant Features) giua am thanh phong thu va tieng ngan nga.

In [ ]:
from torch.autograd import Function

class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_val: float):
        ctx.lambda_val = lambda_val
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    """Lớp đảo ngược gradient với hệ số lambda điều chỉnh động theo epoch."""
    def __init__(self, lambda_val: float = 0.2):
        super(GradientReversalLayer, self).__init__()
        self.lambda_val = lambda_val

    def set_lambda(self, lambda_val: float):
        self.lambda_val = lambda_val

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return GradientReversalFunction.apply(x, self.lambda_val)

### 5.2 Dinh nghia bo phan loai mien cuc bo (LocalDomainClassifier)
Phan loai mien am thanh truc tiep tren chuoi dac trung cuc bo [B, T=20, 128] bang kien truc Conv1D da tang va LayerNorm. Khac phuc han che cua phien ban cu von chi nhan vector embedding toan cuc 128-D.

In [ ]:
class LocalDomainClassifier(nn.Module):
    """
    Bộ phân loại miền cục bộ (Local Domain Classifier) theo kiến trúc ByteHum:
    - Nhận đầu vào: Chuỗi đặc trưng cục bộ x [B, T, 128] từ Conformer encoder.
    - Áp dụng các tầng tích chập 1D theo trục thời gian để học đặc trưng miền cục bộ.
    - Dự đoán phân phối nhãn miền (0: Vocal phong thu, 1: Tiếng humming) trên từng phân đoạn thời gian.
    """
    def __init__(self, feature_dim: int = 128, hidden_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(feature_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, 128] -> transpose thành [B, 128, T] cho Conv1D
        x_t = x.transpose(1, 2)
        feat = self.conv(x_t) # [B, hidden_dim, T]
        # Frame-level logits: [B, T]
        frame_logits = self.fc(feat.transpose(1, 2)).squeeze(-1)
        # Tổng hợp logits qua trục thời gian T
        return frame_logits.mean(dim=-1) # [B]

### 5.3 Ham nap mo hinh nen tang CoverHunter Conformer tu checkpoint Stage 1
Nap tron ven 265/265 tensor tu checkpoint SHS100K pre-trained (6 khoi Conformer, Conv2d Subsampling 4x, Attention Pooling 128-D).

In [ ]:
def build_coverhunter_model(ckpt_path: str = None, device: str = "cpu", allow_random_init: bool = False):
    """
    Khởi tạo mô hình CoverHunter Conformer chính thức (9.91M tham số, 6 khối Conformer, Attention Pooling 128-D).
    Nạp trọn vẹn 265/265 tensor từ checkpoint SHS100K pre-trained.
    """
    # Cấu hình siêu tham số mặc định chuẩn xác của CoverHunter SHS100K
    hp = {
        'input_dim': 96, 'embed_dim': 128,
        'encoder': {'output_dims': 128, 'num_blocks': 6, 'attention_dim': 256},
        'pool_type': 'attention',
        'ce': {'output_dims': 30000, 'weight': 1.0, 'gamma': 2},
        'triplet': {'margin': 0.3, 'weight': 0.1},
        'center': {'weight': 0.0}
    }

    ch_model = Model(hp)

    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state_dict = ckpt.get("generator", ckpt.get("model_state_dict", ckpt))
        ch_model.load_state_dict(state_dict)
        total_p = sum(p.numel() for p in ch_model.parameters())
        print(f"[+] Nạp thành công 265/265 tensors ({total_p:,} params) từ CoverHunter Pretrained:\n    -> {ckpt_path}")
    elif allow_random_init:
        print("[*] Khoi tao backbone tam thoi; checkpoint fine-tuned se duoc nap ngay sau do.")
    else:
        raise FileNotFoundError(f"Bat buoc phai co checkpoint Stage 1 SHS100K: {ckpt_path}")

    ch_model = ch_model.to(device)
    return ch_model

### 5.4 Dinh nghia mo hinh tich hop Conformer-ByteHum voi Local Features va LAC
Ket hop Conformer backbone voi bo phan loai can chinh cuc bo Local Alignment Classifier (LAC) qua phep toan True MaxMean va bo phan loai mien cuc bo qua GRL. Ham forward tra ve dong thoi Global Embedding [B, 128] va chuoi Local Features [B, T=20, 128].

In [ ]:
class LocalAlignmentClassifier(nn.Module):
    """
    Local Alignment Classifier (LAC) theo kiến trúc ByteHum (ICASSP 2024):
    - Trọng số Prototype: W in R^{K x L x C}, trong đó K là số lớp bài hát (train songs),
      L là số phân đoạn prototype cục bộ (mặc định L=4), C=128.
    - Điểm tương đồng MaxMean: S(X, W_k) = (1/L) * sum_{l=1}^L max_{t=1}^T cos(x_t, w_{k,l}).
    - Cho phép căn chỉnh cục bộ bất biến thời gian và trôi dạt nhịp điệu giữa Humming và Vocal.
    """
    def __init__(self, num_classes: int = 210, feature_dim: int = 128, num_prototypes: int = 4):
        super().__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_prototypes = num_prototypes
        # Khởi tạo ma trận prototype W [K, L, C]
        self.prototypes = nn.Parameter(torch.randn(num_classes, num_prototypes, feature_dim) * 0.05)
        self.temperature = nn.Parameter(torch.tensor(10.0))

    def forward(self, local_feat: torch.Tensor) -> torch.Tensor:
        # local_feat: [B, T, 128]
        # Chuẩn hóa L2 trên trục đặc trưng 128-D
        x_norm = F.normalize(local_feat, p=2, dim=-1) # [B, T, C]
        w_norm = F.normalize(self.prototypes, p=2, dim=-1) # [K, L, C]

        # Tính cosine similarity giữa toàn bộ T frames và K*L prototypes qua phép nhân ma trận:
        w_flat = w_norm.view(-1, self.feature_dim).t() # [C, K*L]
        sim = torch.matmul(x_norm, w_flat) # [B, T, K*L]
        B, T, _ = sim.shape
        sim = sim.view(B, T, self.num_classes, self.num_prototypes) # [B, T, K, L]

        # Phep toan MaxMean: Max theo truc thoi gian T, Mean theo so prototypes L
        max_t, _ = sim.max(dim=1) # [B, K, L]
        logits = max_t.mean(dim=-1) # [B, K]
        return logits * self.temperature.clamp(min=1.0, max=100.0)

class ConformerByteHumModel(nn.Module):
    """
    MÔ HÌNH HOÀN CHỈNH CONFORMER-BYTEHUM CHO QBH DOMAIN ADAPTATION:
    - Backbone: Conformer (6 khối, Conv2d Subsampling 4x -> T_enc=20 frames cho 10s audio)
    - Output 1: global_emb [B, 128] (qua Attentive Statistics Pooling + L2 Normalize)
    - Output 2: local_feat [B, T=20, 128] (chuỗi đặc trưng cục bộ trước pooling)
    - Output 3: lac_logits [B, K] (Local Alignment Classifier phân loại bài hát qua MaxMean)
    - Output 4: domain_logits [B] (Local Domain Classifier phân biệt miền Vocal vs Humming)
    """
    def __init__(self, ckpt_path: str = None, input_dim: int = 96, embedding_dim: int = 128, num_classes: int = 210, lambda_grl: float = 0.2, allow_random_init: bool = False):
        super().__init__()
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        if embedding_dim not in (128, 256, 512):
            raise ValueError('embedding_dim must be 128, 256 or 512')
        self.local_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)
        self.global_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)

        # Nạp backbone Conformer pre-trained từ CoverHunter Stage 1
        self.feature_extractor = build_coverhunter_model(ckpt_path=ckpt_path, allow_random_init=allow_random_init)
        
        # Đóng băng các tầng phụ không dùng của Stage 1 (ce_layer, center_loss)
        if hasattr(self.feature_extractor, '_ce_layer'):
            for p in self.feature_extractor._ce_layer.parameters(): p.requires_grad = False
        if hasattr(self.feature_extractor, '_center_loss'):
            for p in self.feature_extractor._center_loss.parameters(): p.requires_grad = False

        self.grl = GradientReversalLayer(lambda_val=lambda_grl)
        self.domain_classifier = LocalDomainClassifier(feature_dim=embedding_dim, hidden_dim=64)
        self.lac = LocalAlignmentClassifier(num_classes=num_classes, feature_dim=embedding_dim, num_prototypes=4)

    def train(self, mode: bool = True):
        super().train(mode)
        # Preserve the SHS100K CMVN statistics while fine-tuning on humming audio.
        if mode and hasattr(self.feature_extractor, '_global_cmvn'):
            self.feature_extractor._global_cmvn.eval()
        return self

    def set_lambda(self, lambda_val: float):
        self.grl.set_lambda(lambda_val)

    def _prepare_input(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 4: x = x.squeeze(1)
        if x.dim() == 3:
            if x.shape[1] == 96 and x.shape[2] != 96: x = x.transpose(1, 2)
        elif x.dim() == 2:
            if x.shape[0] == 96 and x.shape[1] != 96: x = x.t().unsqueeze(0)
            else: x = x.unsqueeze(0)
        return x

    def extract_backbone_features(self, x: torch.Tensor) -> torch.Tensor:
        """Trích xuất chuỗi đặc trưng cục bộ [B, T=20, 128] từ Conformer encoder trước ASP pooling."""
        x_prep = self._prepare_input(x)
        x_cmvn = self.feature_extractor._global_cmvn(x_prep.transpose(1, 2)).transpose(1, 2)
        xs_lens = torch.full([x_cmvn.size(0)], fill_value=x_cmvn.size(1), dtype=torch.long, device=x_cmvn.device)
        x_enc, _ = self.feature_extractor._encoder(x_cmvn, xs_lens=xs_lens, decoding_chunk_size=-1)
        return x_enc # [B, T=20, 128]

    def extract_local_features(self, x):
        return self.local_projection(self.extract_backbone_features(x))

    def extract_global_embedding(self, local_feat: torch.Tensor) -> torch.Tensor:
        """Chiếu chuỗi đặc trưng cục bộ qua Attentive Statistics Pooling thành Global 128-D Embedding."""
        emb = self.feature_extractor._pool_layer(local_feat)
        return F.normalize(self.global_projection(emb), p=2, dim=-1)

    def extract_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Hàm tiện ích trích xuất trực tiếp Global Embedding 128-D từ phổ CQT đầu vào."""
        return self.extract_global_embedding(self.extract_backbone_features(x))

    def forward(self, x: torch.Tensor, compute_lac: bool = True, compute_domain: bool = True):
        """Return global/local features and only the auxiliary heads requested by the run."""
        backbone_feat = self.extract_backbone_features(x)
        local_feat = self.local_projection(backbone_feat)
        global_emb = self.extract_global_embedding(backbone_feat)
        lac_logits = self.lac(local_feat) if compute_lac else None
        domain_logits = self.domain_classifier(self.grl(local_feat)) if compute_domain else None
        return global_emb, local_feat, lac_logits, domain_logits

    def freeze_lower_layers(self, num_blocks_to_freeze: int = 3, freeze_frontend: bool = True):
        """Đóng băng tầng frontend và các khối Conformer đầu tiên theo chiến lược Deep Freeze."""
        if freeze_frontend:
            if hasattr(self.feature_extractor, '_global_cmvn'):
                for p in self.feature_extractor._global_cmvn.parameters(): p.requires_grad = False
            if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'embed'):
                for p in self.feature_extractor._encoder.embed.parameters(): p.requires_grad = False

        if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'encoders'):
            for i, block in enumerate(self.feature_extractor._encoder.encoders):
                if i < num_blocks_to_freeze:
                    for p in block.parameters(): p.requires_grad = False
                else:
                    for p in block.parameters(): p.requires_grad = True

        if hasattr(self.feature_extractor, '_pool_layer'):
            for p in self.feature_extractor._pool_layer.parameters(): p.requires_grad = True
        for p in self.domain_classifier.parameters(): p.requires_grad = True
        for p in self.lac.parameters(): p.requires_grad = True

### Initialize the pretrained model
LOCAL_FREEZE_BLOCKS controls the frozen prefix (default: first 3 blocks).


In [ ]:
# Change RUN_NAME, then restart the kernel for each independently trained ablation.
EMBEDDING_DIM = 128
RUN_NAME = f'accuracy_posmix{round(LOCAL_POSITIVE_VOCAL_RATIO * 100):02d}'
ABLATIONS = {'full': dict(global_triplet=1., lac=1., lat=1., grl=.2, memory=True),
             RUN_NAME: dict(global_triplet=1., lac=1., lat=1., grl=.2, memory=True)}
for component in ('global_triplet','lac','lat','grl','memory'):
    ABLATIONS['without_' + component] = dict(ABLATIONS['full'], **{component: False if component == 'memory' else 0.})
VALIDATION_GALLERY_MODE = os.environ.get('QBH_VALIDATION_GALLERY', 'val').strip().lower()
RUN_CONFIG = dict(ABLATIONS[RUN_NAME], embedding_dim=EMBEDDING_DIM, time_grid_distortion=True, prototype_diversity=0.05, pipeline_version=4, cqt_floor_db=-80.0, validation_gallery=VALIDATION_GALLERY_MODE)
RUN_CONFIG.update(local_profile='4gb', microbatch=BATCH_SIZE, accumulation=LOCAL_ACCUMULATION_STEPS,
                   freeze_blocks=LOCAL_FREEZE_BLOCKS, num_epochs=LOCAL_NUM_EPOCHS,
                   songs_per_batch=SONGS_PER_BATCH, samples_per_song=SAMPLES_PER_SONG,
                   positive_vocal_ratio=LOCAL_POSITIVE_VOCAL_RATIO, initialization_checkpoint=str(INITIAL_LOCAL_CHECKPOINT))
RUN_TAG = f'{RUN_NAME}_d{EMBEDDING_DIM}_timegrid_lat_v4_local4gb'
OUTPUT_DIR = str(STAGE2_ROOT / 'runs' / 'accuracy_finetune' / RUN_TAG)
CHECKPOINT_DIR = str(Path(OUTPUT_DIR) / 'checkpoints')
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
NUM_TRAIN_SONGS = len(train_dataset.songs)

model = ConformerByteHumModel(
    ckpt_path=STAGE1_CKPT,
    input_dim=96,
    embedding_dim=EMBEDDING_DIM,
    num_classes=NUM_TRAIN_SONGS,
    lambda_grl=0.2
)

# Thực thi Deep Freeze 3 khối Conformer đầu tiên
model.freeze_lower_layers(num_blocks_to_freeze=LOCAL_FREEZE_BLOCKS, freeze_frontend=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

if not INITIAL_LOCAL_CHECKPOINT.is_file():
    raise FileNotFoundError(INITIAL_LOCAL_CHECKPOINT)
initial_payload = torch.load(INITIAL_LOCAL_CHECKPOINT, map_location='cpu', weights_only=False)
if initial_payload.get('run_config', {}).get('pipeline_version') != 4:
    raise ValueError('Initial checkpoint is not a pipeline-v4 model.')
if initial_payload.get('train_songs') != train_dataset.songs:
    raise ValueError('Initial checkpoint train-song mapping does not match this dataset.')
model.load_state_dict(initial_payload['model_state_dict'], strict=True)
print('Initialized from existing local best:', INITIAL_LOCAL_CHECKPOINT)
del initial_payload

# Thống kê chi tiết số lượng tham số
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

# Tính tỷ lệ đóng băng trên riêng phần Backbone Conformer
backbone_params = [(name, p) for name, p in model.feature_extractor.named_parameters() if '_ce_layer' not in name and '_center_loss' not in name]
bb_frozen = sum(p.numel() for _, p in backbone_params if not p.requires_grad)
bb_trainable = sum(p.numel() for _, p in backbone_params if p.requires_grad)
bb_total = bb_frozen + bb_trainable
bb_freeze_ratio = (bb_frozen / bb_total) * 100 if bb_total > 0 else 0.0

print("[+] KHỞI TẠO VÀ THIẾT LẬP DEEP FREEZE MÔ HÌNH THÀNH CÔNG:")
print(f"    - Thiết bị thực thi              : {device}")
print(f"    - Tổng tham số mô hình           : {total_params:,}")
print(f"    - Tham số cho phép huấn luyện    : {trainable_params:,}")
print(f"    - Tham số Backbone đóng băng     : {bb_frozen:,} / {bb_total:,} ({bb_freeze_ratio:.2f}%)")
print(f"    - Số lớp bài hát LAC (Classes)   : {NUM_TRAIN_SONGS} (Mỗi bài 4 Prototypes 128-D)")

# Kiểm tra forward pass trên 1 batch mẫu từ train_loader
model.eval()
with torch.no_grad():
    sample_cqt = sample_batch['anchor'].to(device) # [B, 1, 96, 83]
    g_emb, l_feat, lac_logits, dom_logits = model(sample_cqt)

print("\n[+] KIỂM TRA ĐẦU RA 4 TENSOR CỦA MÔ HÌNH CONFORMER-BYTEHUM:")
print(f"    - Global Embedding shape        : {g_emb.shape} (Kỳ vọng: [{BATCH_SIZE}, 128])")
print(f"    - Local Feature Sequence shape  : {l_feat.shape} (Kỳ vọng: [{BATCH_SIZE}, 20, 128] - 4x Subsampling)")
print(f"    - LAC Similarity Logits shape   : {lac_logits.shape} (Kỳ vọng: [{BATCH_SIZE}, {NUM_TRAIN_SONGS}])")
print(f"    - Local Domain Pred Logits shape: {dom_logits.shape} (Kỳ vọng: [{BATCH_SIZE}])")

assert g_emb.shape == (BATCH_SIZE, EMBEDDING_DIM), f"Lỗi shape Global Embedding: {g_emb.shape}"
assert l_feat.shape == (BATCH_SIZE, 20, EMBEDDING_DIM), f"Lỗi shape Local Features: {l_feat.shape}"
assert lac_logits.shape == (BATCH_SIZE, NUM_TRAIN_SONGS), f"Lỗi shape LAC Logits: {lac_logits.shape}"
assert dom_logits.shape == (BATCH_SIZE,), f"Lỗi shape Domain Logits: {dom_logits.shape}"
print("[+] Tất cả các đầu ra đều khớp chính xác 100% với đặc tả thiết kế!")
del sample_cqt, g_emb, l_feat, lac_logits, dom_logits
torch.cuda.empty_cache()


### 6.1 Ham mat mat da nhiem va Hard Negative Mining
Cosine Triplet Margin Loss (margin=0.20) dung song-balanced batch, semi-hard/hard curriculum, Top-5 aggregation va FIFO cross-batch memory bank 4.096 embeddings. Hai thanh phan con lai la Domain BCE qua GRL va Cross-Entropy tren Local Alignment Classifier (LAC).

In [ ]:
class CrossBatchMemoryBank:
    """FIFO memory bank tren CPU; embeddings trong bank khong mang gradient."""
    def __init__(self, capacity=4096):
        self.capacity = int(capacity)
        self.embeddings = None
        self.song_ids = None

    def __len__(self):
        return 0 if self.song_ids is None else int(self.song_ids.numel())

    def clear(self):
        self.embeddings = None
        self.song_ids = None

    @torch.no_grad()
    def enqueue(self, embeddings, song_ids):
        new_embeddings = F.normalize(embeddings.detach(), p=2, dim=-1).cpu()
        new_song_ids = song_ids.detach().long().cpu()
        if self.embeddings is None:
            self.embeddings = new_embeddings[-self.capacity:]
            self.song_ids = new_song_ids[-self.capacity:]
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)[-self.capacity:]
            self.song_ids = torch.cat([self.song_ids, new_song_ids], dim=0)[-self.capacity:]

    def candidates(self, device, dtype):
        if self.embeddings is None:
            return None, None
        return self.embeddings.to(device=device, dtype=dtype), self.song_ids.to(device=device)


def mining_hardest_ratio(epoch):
    """Curriculum: semi-hard first, then gradually emphasize hardest negatives."""
    if epoch <= 5:
        return 0.0
    if epoch <= 15:
        return 0.3
    return 0.7


def _topk_masked_mean(values, mask, top_k):
    masked = values.masked_fill(~mask, float('-inf'))
    k = min(int(top_k), masked.shape[1])
    top_values = torch.topk(masked, k=k, dim=1).values
    finite = torch.isfinite(top_values)
    total = torch.where(finite, top_values, torch.zeros_like(top_values)).sum(dim=1, keepdim=True)
    count = finite.sum(dim=1, keepdim=True)
    return total / count.clamp_min(1), count > 0


def in_batch_cosine_triplet_loss(
    emb_a,
    emb_p,
    emb_n,
    song_ids,
    neg_song_ids,
    margin=0.20,
    memory_bank=None,
    hardest_ratio=0.0,
    top_k=5
):
    """Semi-hard/hard curriculum over in-batch candidates plus a cross-batch memory bank."""
    cos_pos = torch.sum(emb_a * emb_p, dim=-1, keepdim=True)
    candidate_embeddings = [emb_n, emb_p, emb_a]
    candidate_song_ids = [neg_song_ids, song_ids, song_ids]

    if memory_bank is not None:
        memory_embeddings, memory_song_ids = memory_bank.candidates(emb_a.device, emb_a.dtype)
        if memory_embeddings is not None:
            candidate_embeddings.append(memory_embeddings)
            candidate_song_ids.append(memory_song_ids)

    candidates = torch.cat(candidate_embeddings, dim=0)
    candidate_labels = torch.cat(candidate_song_ids, dim=0)
    similarities = torch.mm(emb_a, candidates.T)
    valid_negative = song_ids.unsqueeze(1) != candidate_labels.unsqueeze(0)

    hard_similarity, has_hard = _topk_masked_mean(similarities, valid_negative, top_k)
    semi_hard_mask = (
        valid_negative
        & (similarities < cos_pos)
        & (similarities > (cos_pos - margin))
    )
    semi_similarity, has_semi = _topk_masked_mean(similarities, semi_hard_mask, top_k)
    semi_or_fallback = torch.where(has_semi, semi_similarity, hard_similarity)
    selected_negative = (1.0 - hardest_ratio) * semi_or_fallback + hardest_ratio * hard_similarity

    if not bool(has_hard.all()):
        raise RuntimeError("Batch khong co negative hop le sau khi mask song ID")
    return F.relu(selected_negative - cos_pos + margin).mean()

# Tiêu chuẩn mất mát cho Domain Adaptation và Local Alignment
criterion_domain = nn.BCEWithLogitsLoss()
criterion_lac = nn.CrossEntropyLoss()

print("[+] Loss weights are controlled only by RUN_CONFIG; domain BCE uses weight 1.0 and GRL controls the backbone gradient.")
def prototype_diversity_loss(prototypes):
    """Discourage LAC prototypes of the same song from collapsing to one direction."""
    normalized = F.normalize(prototypes, p=2, dim=-1)
    gram = torch.matmul(normalized, normalized.transpose(-1, -2))
    eye = torch.eye(gram.shape[-1], device=gram.device, dtype=gram.dtype).unsqueeze(0)
    off_diagonal = (gram - eye) * (1.0 - eye)
    return off_diagonal.square().sum() / max(1, gram.shape[0] * gram.shape[1] * (gram.shape[1] - 1))


def local_alignment_triplet(
    anchor, positive, negative, song_ids, negative_song_ids,
    margin=0.2, hardest_ratio=0.0, top_k=5
):
    """Fully vectorized query-directed MaxMean with semi-hard-to-hard curriculum."""
    anchor, positive, negative = [F.normalize(x, dim=-1) for x in (anchor, positive, negative)]
    positive_scores = torch.bmm(anchor, positive.transpose(1, 2)).max(-1).values.mean(-1, keepdim=True)
    candidates = torch.cat([positive, negative], dim=0)
    labels = torch.cat([song_ids, negative_song_ids], dim=0)
    valid = song_ids.unsqueeze(1) != labels.unsqueeze(0)
    if not bool(valid.any(dim=1).all()):
        raise ValueError('LAT requires different-song negatives for every anchor')

    similarities = torch.einsum('btc,nsc->bnts', anchor, candidates).max(dim=-1).values.mean(dim=-1)
    hard_score, has_hard = _topk_masked_mean(similarities, valid, top_k)
    semi_mask = valid & (similarities < positive_scores) & (similarities > positive_scores - margin)
    semi_score, has_semi = _topk_masked_mean(similarities, semi_mask, top_k)
    semi_or_fallback = torch.where(has_semi, semi_score, hard_score)
    selected = (1.0 - hardest_ratio) * semi_or_fallback + hardest_ratio * hard_score
    if not bool(has_hard.all()):
        raise RuntimeError('LAT has no valid negative')
    return F.relu(selected - positive_scores + margin).mean()


### Optimizer and local training schedule
FP16 AMP and gradient accumulation reduce training VRAM.


In [ ]:
# Phân tầng tốc độ học (Differential Learning Rates):
# - Tầng Backbone (Conformer block 3-5 + ASP): LR vừa phải (1e-4) để giữ tri thức pre-trained
# - Tầng đầu mới (LAC + Local Domain Classifier): LR cao hơn (3e-4) để hội tụ nhanh
optimizer = torch.optim.AdamW([
    {'params': [p for name, p in model.named_parameters() if not any(k in name for k in ['domain_classifier', 'lac']) and p.requires_grad], 'lr': 1e-4},
    {'params': [p for name, p in model.named_parameters() if any(k in name for k in ['domain_classifier', 'lac']) and p.requires_grad], 'lr': 3e-4}
], weight_decay=1e-2)

NUM_EPOCHS = LOCAL_NUM_EPOCHS
EARLY_STOPPING_PATIENCE = 5
GRAD_ACCUMULATION_STEPS = LOCAL_ACCUMULATION_STEPS
TRIPLET_MARGIN = 0.20
MAX_GRAD_NORM = 1.0
MEMORY_BANK_CAPACITY = 4096
MINING_TOP_K = 5
STEPS_PER_EPOCH = len(train_loader)
OPTIMIZER_STEPS_PER_EPOCH = (STEPS_PER_EPOCH + GRAD_ACCUMULATION_STEPS - 1) // GRAD_ACCUMULATION_STEPS
VAL_STEPS = len(val_loader)
TOTAL_STEPS = NUM_EPOCHS * OPTIMIZER_STEPS_PER_EPOCH

negative_memory_bank = CrossBatchMemoryBank(capacity=MEMORY_BANK_CAPACITY)
AMP_ENABLED = device.type == 'cuda'
amp_scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED, init_scale=1024.0)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[1e-4, 3e-4], total_steps=TOTAL_STEPS, pct_start=0.10, anneal_strategy='cos'
)

history = {
    'train_triplet_loss': [], 'train_domain_loss': [], 'train_lac_loss': [], 'train_total_loss': [],
    'val_triplet_loss': [], 'val_domain_loss': [], 'val_lac_loss': [], 'val_total_loss': [],
    'val_mrr': [], 'val_local_mrr': [], 'val_global_mrr': [],
    'train_lat_loss': [], 'val_lat_loss': [], 'train_prototype_loss': [],
    'train_retrieval_loss': []
}

best_val_mrr = 0.0
best_val_loss = float('inf')
finetuned_ckpt_path = os.path.join(CHECKPOINT_DIR, f"conformer_bytehum_{RUN_TAG}_best.pth")
last_ckpt_path = os.path.join(CHECKPOINT_DIR, f"conformer_bytehum_{RUN_TAG}_last.pth")

FORCE_RETRAIN = False  # Đặt True khi chủ động muốn huấn luyện lại và ghi đè checkpoint tốt nhất
print(f"[+] Thiết lập Optimizer và Lịch trình huấn luyện hoàn tất:")
print(f"    - Epochs: {NUM_EPOCHS} | Batches/Epoch: {STEPS_PER_EPOCH} | Optimizer Steps/Epoch: {OPTIMIZER_STEPS_PER_EPOCH} | Total Steps: {TOTAL_STEPS}")
print(f"    - Mining: P={SONGS_PER_BATCH}, K={SAMPLES_PER_SONG}, Top-{MINING_TOP_K}, Memory={MEMORY_BANK_CAPACITY}")
print(f"    - Checkpoint lưu trữ: {finetuned_ckpt_path}")

### 6.3 Vong lap huan luyen va kiem dinh mo hinh (Training & Validation Loop)
Huan luyen da nhiem voi GRL dong va gradient clipping norm 5.0. Kiem dinh sau moi epoch bang validation loss va Retrieval MRR tren toan bo gallery validation. Luu Best Checkpoint theo MRR truy van de toi uu truc tiep cho he thong QbH.

In [ ]:
from tqdm import tqdm
@torch.no_grad()
def _vectorized_local_maxmean(query_local, gallery_padded, gallery_mask, song_batch_size=LOCAL_GALLERY_BLOCK_SIZE):
    """Keep gallery features on CPU and transfer bounded candidate blocks to the query device."""
    score_batches = []
    for offset in range(0, gallery_padded.shape[0], song_batch_size):
        candidates = gallery_padded[offset:offset + song_batch_size].to(query_local.device, non_blocking=True)
        valid = gallery_mask[offset:offset + song_batch_size].to(query_local.device, non_blocking=True)
        similarities = torch.einsum('qd,sgd->sqg', query_local, candidates)
        similarities = similarities.masked_fill(~valid[:, None, :], float('-inf'))
        score_batches.append(similarities.max(dim=-1).values.mean(dim=-1))
    return torch.cat(score_batches, dim=0)


@torch.no_grad()
def _vectorized_global_chunk_scores(query_global, gallery_padded, gallery_mask, song_batch_size=LOCAL_GALLERY_BLOCK_SIZE):
    score_batches = []
    for offset in range(0, gallery_padded.shape[0], song_batch_size):
        candidates = gallery_padded[offset:offset + song_batch_size].to(query_global.device, non_blocking=True)
        valid = gallery_mask[offset:offset + song_batch_size].to(query_global.device, non_blocking=True)
        similarities = torch.einsum('qd,sgd->sqg', query_global, candidates)
        similarities = similarities.masked_fill(~valid[:, None, :], float('-inf'))
        score_batches.append(similarities.amax(dim=(1, 2)))
    return torch.cat(score_batches, dim=0)


def validation_gallery_paths(dataset):
    mode = VALIDATION_GALLERY_MODE
    if mode not in {'val', 'full'}:
        raise ValueError("QBH_VALIDATION_GALLERY must be 'val' or 'full'")
    if mode == 'val':
        gallery = {}
        for song, fragments in dataset.song_data.items():
            paths = []
            for pool in fragments.values():
                paths.extend(pool['refs'] if pool['refs'] else pool['vocals'])
            gallery[song] = paths
        return gallery

    manifest_path = os.environ.get('QBH_GALLERY_MANIFEST')
    if manifest_path:
        import json
        with open(manifest_path, encoding='utf-8') as stream:
            manifest = json.load(stream)
        gallery = {str(song): ([paths] if isinstance(paths, str) else list(paths)) for song, paths in manifest.items()}
    else:
        if not MUSIC_DB_DIR or not os.path.isdir(MUSIC_DB_DIR):
            raise FileNotFoundError(f"Full validation gallery is unavailable: {MUSIC_DB_DIR}")
        extensions = ('.wav', '.mp3', '.flac', '.ogg', '.m4a')
        gallery = {os.path.splitext(name)[0]: [os.path.join(MUSIC_DB_DIR, name)]
                   for name in sorted(os.listdir(MUSIC_DB_DIR)) if name.lower().endswith(extensions)}
        print('[!] QBH_GALLERY_MANIFEST is not set; full-gallery IDs are inferred from filenames.')
    missing = sorted({song for song, _, _ in dataset.humming_samples} - set(gallery))
    if missing:
        raise RuntimeError(f"Full gallery thieu {len(missing)} validation songs, vi du: {missing[:3]}")
    return gallery


@torch.no_grad()
def _encode_cqt_list(eval_model, cqts, batch_size=LOCAL_EVAL_BATCH_SIZE):
    globals_out, locals_out = [], []
    for offset in range(0, len(cqts), batch_size):
        tensors = torch.stack(cqts[offset:offset + batch_size], dim=0).to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ENABLED):
            global_features, local_features, _, _ = eval_model(tensors, compute_lac=False, compute_domain=False)
        globals_out.append(F.normalize(global_features.float(), p=2, dim=-1).cpu())
        locals_out.append(F.normalize(local_features.float(), p=2, dim=-1).cpu())
    return torch.cat(globals_out), torch.cat(locals_out)


def _average_tie_rank(scores, target):
    target_score = scores[target]
    greater = int((scores > target_score).sum().item())
    tied = int(torch.isclose(scores, target_score, rtol=1e-6, atol=1e-7).sum().item())
    return 1.0 + greater + 0.5 * max(0, tied - 1), tied


@torch.no_grad()
def compute_validation_retrieval_mrr(eval_model, dataset, query_batch_size=LOCAL_QUERY_BATCH_SIZE):
    """Memory-bounded, batched-query local/global MRR with average tie ranks."""
    eval_model.eval()
    gallery_local, gallery_global = {}, {}
    for song, source_paths in validation_gallery_paths(dataset).items():
        reference_cqts = [chunk for path in source_paths for chunk in dataset._get_raw_cqts(path)]
        if not reference_cqts:
            raise RuntimeError(f"Validation gallery thieu reference cho {song}")
        global_features, local_features = _encode_cqt_list(eval_model, reference_cqts)
        gallery_local[song] = local_features.reshape(-1, local_features.shape[-1])
        gallery_global[song] = global_features

    gallery_songs = list(gallery_global)
    song_to_gallery_idx = {song: idx for idx, song in enumerate(gallery_songs)}
    local_lengths = torch.tensor([gallery_local[song].shape[0] for song in gallery_songs])
    local_padded = nn.utils.rnn.pad_sequence([gallery_local[song] for song in gallery_songs], batch_first=True)
    local_mask = torch.arange(local_padded.shape[1])[None, :] < local_lengths[:, None]
    global_lengths = torch.tensor([gallery_global[song].shape[0] for song in gallery_songs])
    global_padded = nn.utils.rnn.pad_sequence([gallery_global[song] for song in gallery_songs], batch_first=True)
    global_mask = torch.arange(global_padded.shape[1])[None, :] < global_lengths[:, None]

    local_rr, global_rr, fused_rr, tie_queries = [], [], [], 0
    queries = dataset.humming_samples
    for batch_start in range(0, len(queries), query_batch_size):
        records = queries[batch_start:batch_start + query_batch_size]
        chunk_groups = [dataset._get_raw_cqts(path) for _, _, path in records]
        chunk_counts = [len(group) for group in chunk_groups]
        batch_global, batch_local = _encode_cqt_list(eval_model, [chunk for group in chunk_groups for chunk in group])
        global_groups = torch.split(batch_global.to(device), chunk_counts)
        local_groups = torch.split(batch_local.to(device), chunk_counts)
        for (song, _, _), query_global, query_local in zip(records, global_groups, local_groups):
            query_local = query_local.reshape(-1, query_local.shape[-1])
            local_scores = _vectorized_local_maxmean(query_local, local_padded, local_mask)
            global_scores = _vectorized_global_chunk_scores(query_global, global_padded, global_mask)
            target = song_to_gallery_idx[song]
            local_rank, local_ties = _average_tie_rank(local_scores, target)
            global_rank, global_ties = _average_tie_rank(global_scores, target)
            fused_rank, fused_ties = _average_tie_rank(0.5 * (local_scores + global_scores), target)
            tie_queries += int(local_ties > 1 or global_ties > 1 or fused_ties > 1)
            local_rr.append(1.0 / local_rank)
            global_rr.append(1.0 / global_rank)
            fused_rr.append(1.0 / fused_rank)

    if not local_rr:
        raise RuntimeError("Validation retrieval khong co query hop le")
    local_mrr, global_mrr, fused_mrr = float(np.mean(local_rr)), float(np.mean(global_rr)), float(np.mean(fused_rr))
    return {'mrr': fused_mrr, 'fused_mrr': fused_mrr, 'local_mrr': local_mrr,
            'global_mrr': global_mrr, 'queries': len(local_rr), 'gallery_songs': len(gallery_songs),
            'tie_queries': tie_queries}

RESUME_TRAINING = os.environ.get('QBH_RESUME_TRAINING', '0') == '1'
start_epoch = 1
epochs_without_improvement = 0
if RESUME_TRAINING and not LOCAL_EVALUATE_ONLY and not FORCE_RETRAIN and os.path.exists(last_ckpt_path):
    resume_data = torch.load(last_ckpt_path, map_location='cpu', weights_only=False)
    if resume_data.get('run_config') != RUN_CONFIG:
        raise ValueError('Last checkpoint configuration mismatch')
    model.load_state_dict(resume_data['model_state_dict'])
    optimizer.load_state_dict(resume_data['optimizer_state_dict'])
    scheduler.load_state_dict(resume_data['scheduler_state_dict'])
    amp_scaler.load_state_dict(resume_data['scaler_state_dict'])
    history = resume_data['history']
    best_val_mrr = resume_data['best_val_mrr']
    best_val_loss = resume_data['best_val_loss']
    start_epoch = int(resume_data['epoch']) + 1
    random.setstate(resume_data['python_rng_state'])
    np.random.set_state(resume_data['numpy_rng_state'])
    torch.set_rng_state(resume_data['torch_rng_state'])
    loader_generator.set_state(resume_data['loader_rng_state'])
    epochs_without_improvement = resume_data['epochs_without_improvement']
    if resume_data['train_songs'] != train_dataset.songs:
        raise ValueError('Resume train-song mapping mismatch')
    if torch.cuda.is_available() and resume_data.get('cuda_rng_state') is not None:
        torch.cuda.set_rng_state_all(resume_data['cuda_rng_state'])
    del resume_data
    print(f"[+] Resume training from epoch {start_epoch}")

if LOCAL_EVALUATE_ONLY and not os.path.exists(finetuned_ckpt_path):
    raise FileNotFoundError(finetuned_ckpt_path)
if LOCAL_EVALUATE_ONLY or (not RESUME_TRAINING and not FORCE_RETRAIN and os.path.exists(finetuned_ckpt_path)):
    print(f"[*] ĐÃ PHÁT HIỆN CHECKPOINT TỐI ƯU CÓ SẴN TẠI: {finetuned_ckpt_path}")
    saved_data = torch.load(finetuned_ckpt_path, map_location='cpu', weights_only=False)
    if saved_data.get('run_config') != RUN_CONFIG:
        raise ValueError('Checkpoint configuration mismatch')
    model.load_state_dict(saved_data['model_state_dict'])
    if 'history' in saved_data: history = saved_data['history']
    print(f"[+] Đã nạp thành công trọng số tinh chỉnh tối ưu (Best Val MRR: {saved_data.get('best_val_mrr', 0.0):.4f})!")
else:
    print("=" * 105)
    print(f"{'Epoch':^7} | {'Train Tot':^11} | {'Tr Trip':^9} | {'Tr Dom':^8} | {'Tr LAC':^8} | {'Val Tot':^10} | {'Val MRR':^9} | {'Thời Gian':^9}")
    print("=" * 105)
    
    epoch_range = range(start_epoch, NUM_EPOCHS + 1) if epochs_without_improvement < EARLY_STOPPING_PATIENCE else []
    for epoch in epoch_range:
        t0_ep = time.time()
        hardest_ratio = mining_hardest_ratio(epoch)
        
        # ------------------ 1. HUẤN LUYỆN ĐA NHIỆM ------------------
        model.train()
        if RUN_CONFIG['memory']:
            negative_memory_bank.clear()
        tr_trip_list, tr_dom_list, tr_lac_list, tr_tot_list = [], [], [], []
        tr_lat_list, tr_proto_list, tr_retrieval_list = [], [], []
        
        optimizer.zero_grad(set_to_none=True)
        train_pbar = tqdm(train_loader, total=STEPS_PER_EPOCH, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [Train]", leave=False)
        for step_idx, batch in enumerate(train_pbar):
            try:
                # Cập nhật Dynamic GRL Lambda theo lịch trình DANN
                current_optimizer_step = (epoch - 1) * OPTIMIZER_STEPS_PER_EPOCH + (step_idx // GRAD_ACCUMULATION_STEPS)
                p = float(current_optimizer_step) / float(max(1, TOTAL_STEPS))
                current_lambda = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0
                model.set_lambda(current_lambda * float(RUN_CONFIG['grl']))
                
                a = batch['anchor'].to(device)
                p_t = batch['positive'].to(device)
                n = batch['negative'].to(device)
                songs = batch['song_idx'].to(device)
                neg_songs = batch['negative_song_idx'].to(device)
                dom_a = batch['domain_anchor'].to(device).squeeze(-1)
                dom_p = batch['domain_positive'].to(device).squeeze(-1)
                dom_n = batch['domain_negative'].to(device).squeeze(-1)
                
                # Forward đồng thời anchor, positive, negative với mixed precision trên CUDA.
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ENABLED):
                    use_lac = RUN_CONFIG['lac'] > 0
                    use_domain = RUN_CONFIG['grl'] > 0
                    g_a, l_a, lac_a, dom_a_pred = model(a, compute_lac=use_lac, compute_domain=use_domain)
                    g_p, l_p, lac_p, dom_p_pred = model(p_t, compute_lac=use_lac, compute_domain=use_domain)
                    g_n, l_n, lac_n, dom_n_pred = model(n, compute_lac=use_lac, compute_domain=use_domain)
                g_a, l_a, g_p, l_p, g_n, l_n = [x.float() for x in (g_a, l_a, g_p, l_p, g_n, l_n)]
                if use_lac:
                    lac_a, lac_p, lac_n = [x.float() for x in (lac_a, lac_p, lac_n)]
                if use_domain:
                    dom_a_pred, dom_p_pred, dom_n_pred = [x.float() for x in (dom_a_pred, dom_p_pred, dom_n_pred)]
                
                # 1. Global triplet is skipped entirely in its ablation.
                if RUN_CONFIG['global_triplet'] > 0:
                    l_trip = in_batch_cosine_triplet_loss(
                        g_a, g_p, g_n, song_ids=songs, neg_song_ids=neg_songs,
                        margin=TRIPLET_MARGIN,
                        memory_bank=negative_memory_bank if RUN_CONFIG['memory'] else None,
                        hardest_ratio=hardest_ratio, top_k=MINING_TOP_K
                    )
                else:
                    l_trip = g_a.new_zeros(())

                if RUN_CONFIG['memory'] and RUN_CONFIG['global_triplet'] > 0:
                    negative_memory_bank.enqueue(
                    torch.cat([g_a, g_p, g_n], dim=0),
                    torch.cat([songs, songs, neg_songs], dim=0)
                )

                # 2. Local Domain Adversarial Loss
                if use_domain:
                    dom_logits = torch.cat([dom_a_pred, dom_p_pred, dom_n_pred], dim=0)
                    dom_targets = torch.cat([dom_a, dom_p, dom_n], dim=0)
                    l_dom = criterion_domain(dom_logits, dom_targets)
                else:
                    l_dom = g_a.new_zeros(())

                # 3. LAC and diversity are skipped entirely in their ablation.
                if use_lac:
                    l_lac = (criterion_lac(lac_a, songs) + criterion_lac(lac_p, songs) + criterion_lac(lac_n, neg_songs)) / 3.0
                    l_proto = prototype_diversity_loss(model.lac.prototypes)
                else:
                    l_lac = l_proto = g_a.new_zeros(())
                
                # Tổng hợp hàm mất mát
                if RUN_CONFIG['lat'] > 0:
                    l_lat = local_alignment_triplet(
                        l_a, l_p, l_n, songs, neg_songs, TRIPLET_MARGIN,
                        hardest_ratio=hardest_ratio, top_k=MINING_TOP_K
                    )
                else:
                    l_lat = l_a.new_zeros(())
                loss = (
                    RUN_CONFIG['global_triplet'] * l_trip
                    + l_dom
                    + RUN_CONFIG['lac'] * l_lac
                    + RUN_CONFIG['lat'] * l_lat
                    + RUN_CONFIG['lac'] * RUN_CONFIG['prototype_diversity'] * l_proto
                )
                retrieval_loss = RUN_CONFIG['global_triplet'] * l_trip + l_dom + RUN_CONFIG['lat'] * l_lat
                tr_lat_list.append(l_lat.item())
                tr_proto_list.append(l_proto.item())
                tr_retrieval_list.append(retrieval_loss.item())
                
                accumulation_divisor = min(GRAD_ACCUMULATION_STEPS, STEPS_PER_EPOCH - (step_idx // GRAD_ACCUMULATION_STEPS) * GRAD_ACCUMULATION_STEPS)
                amp_scaler.scale(loss / accumulation_divisor).backward()
                if (step_idx + 1) % GRAD_ACCUMULATION_STEPS == 0 or (step_idx + 1) == STEPS_PER_EPOCH:
                    amp_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
                    scale_before = amp_scaler.get_scale()
                    amp_scaler.step(optimizer)
                    amp_scaler.update()
                    if amp_scaler.get_scale() >= scale_before:
                        scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                
                tr_trip_list.append(l_trip.item())
                tr_dom_list.append(l_dom.item())
                tr_lac_list.append(l_lac.item())
                tr_tot_list.append(loss.item())
                
                train_pbar.set_postfix({
                    'Trip': f"{l_trip.item():.3f}",
                    'Dom': f"{l_dom.item():.3f}",
                    'LAC': f"{l_lac.item():.3f}",
                    'Tot': f"{loss.item():.3f}",
                    'HardMix': f"{hardest_ratio:.1f}",
                    'Bank': len(negative_memory_bank)
                })
            except Exception as e:
                raise RuntimeError(f"Lỗi batch train tại epoch={epoch}, step={step_idx}") from e
            
        # ------------------ 2. KIỂM ĐỊNH & ĐÁNH GIÁ TRUY VẤN ------------------
        model.eval()
        va_trip_list, va_dom_list, va_lac_list, va_tot_list = [], [], [], []
        va_lat_list = []
        
        val_pbar = tqdm(val_loader, total=VAL_STEPS, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [Val]", leave=False)
        with torch.no_grad():
            for batch_v in val_pbar:
                try:
                    a_v = batch_v['anchor'].to(device)
                    p_v = batch_v['positive'].to(device)
                    n_v = batch_v['negative'].to(device)
                    songs_v = batch_v['song_idx'].to(device)
                    neg_songs_v = batch_v['negative_song_idx'].to(device)
                    dom_av = batch_v['domain_anchor'].to(device).squeeze(-1)
                    dom_pv = batch_v['domain_positive'].to(device).squeeze(-1)
                    dom_nv = batch_v['domain_negative'].to(device).squeeze(-1)
                    
                    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ENABLED):
                        use_domain = RUN_CONFIG['grl'] > 0
                        g_av, l_av, _, dom_av_pred = model(a_v, compute_lac=False, compute_domain=use_domain)
                        g_pv, l_pv, _, dom_pv_pred = model(p_v, compute_lac=False, compute_domain=use_domain)
                        g_nv, l_nv, _, dom_nv_pred = model(n_v, compute_lac=False, compute_domain=use_domain)
                    g_av, l_av, g_pv, l_pv, g_nv, l_nv = [x.float() for x in (g_av, l_av, g_pv, l_pv, g_nv, l_nv)]
                    if use_domain:
                        dom_av_pred, dom_pv_pred, dom_nv_pred = [x.float() for x in (dom_av_pred, dom_pv_pred, dom_nv_pred)]
                    
                    if RUN_CONFIG['global_triplet'] > 0:
                        l_trip_v = in_batch_cosine_triplet_loss(
                            g_av, g_pv, g_nv, song_ids=songs_v, neg_song_ids=neg_songs_v,
                            margin=TRIPLET_MARGIN, memory_bank=None, hardest_ratio=1.0, top_k=MINING_TOP_K
                        )
                    else:
                        l_trip_v = g_av.new_zeros(())
                    if use_domain:
                        dom_logits_v = torch.cat([dom_av_pred, dom_pv_pred, dom_nv_pred], dim=0)
                        dom_targets_v = torch.cat([dom_av, dom_pv, dom_nv], dim=0)
                        l_dom_v = criterion_domain(dom_logits_v, dom_targets_v)
                    else:
                        l_dom_v = g_av.new_zeros(())
                    # Validation songs unseen khong co proxy LAC tuong ung trong 210 lop train.
                    # Khong tinh CrossEntropy LAC voi nhan val de tranh metric sai nghia.
                    if RUN_CONFIG['lat'] > 0:
                        l_lat_v = local_alignment_triplet(
                            l_av, l_pv, l_nv, songs_v, neg_songs_v, TRIPLET_MARGIN,
                            hardest_ratio=1.0, top_k=MINING_TOP_K
                        )
                    else:
                        l_lat_v = l_av.new_zeros(())
                    loss_v = RUN_CONFIG['global_triplet'] * l_trip_v + l_dom_v + RUN_CONFIG['lat'] * l_lat_v
                    va_lat_list.append(l_lat_v.item())
                    
                    va_trip_list.append(l_trip_v.item())
                    va_dom_list.append(l_dom_v.item())
                    va_tot_list.append(loss_v.item())
                    
                except Exception as e:
                    raise RuntimeError(f"Lỗi validation tại epoch={epoch}") from e
                
        ep_tr_tot = float(np.mean(tr_tot_list)) if tr_tot_list else 0.0
        ep_tr_trip = float(np.mean(tr_trip_list)) if tr_trip_list else 0.0
        ep_tr_dom = float(np.mean(tr_dom_list)) if tr_dom_list else 0.0
        ep_tr_lac = float(np.mean(tr_lac_list)) if tr_lac_list else 0.0
        
        ep_va_tot = float(np.mean(va_tot_list)) if va_tot_list else 0.0
        val_retrieval = compute_validation_retrieval_mrr(model, val_dataset)
        ep_va_mrr = val_retrieval['mrr']
        ep_va_local_mrr = val_retrieval['local_mrr']
        ep_va_global_mrr = val_retrieval['global_mrr']
        
        history['train_lat_loss'].append(float(np.mean(tr_lat_list)))
        history['train_retrieval_loss'].append(float(np.mean(tr_retrieval_list)))
        history['train_prototype_loss'].append(float(np.mean(tr_proto_list)))
        history['val_lat_loss'].append(float(np.mean(va_lat_list)))
        history['train_total_loss'].append(ep_tr_tot)
        history['train_triplet_loss'].append(ep_tr_trip)
        history['train_domain_loss'].append(ep_tr_dom)
        history['train_lac_loss'].append(ep_tr_lac)
        history['val_triplet_loss'].append(float(np.mean(va_trip_list)) if va_trip_list else float('nan'))
        history['val_domain_loss'].append(float(np.mean(va_dom_list)) if va_dom_list else float('nan'))
        history['val_lac_loss'].append(float('nan'))  # LAC CE khong xac dinh cho unseen validation songs.
        history['val_total_loss'].append(ep_va_tot)
        history['val_mrr'].append(ep_va_mrr)
        history['val_local_mrr'].append(ep_va_local_mrr)
        history['val_global_mrr'].append(ep_va_global_mrr)
        
        ep_time = time.time() - t0_ep
        print(f"{epoch:^7} | {ep_tr_tot:^11.4f} | {ep_tr_trip:^9.4f} | {ep_tr_dom:^8.4f} | {ep_tr_lac:^8.4f} | {ep_va_tot:^10.4f} | {ep_va_mrr:^9.4f} | {ep_time:^7.1f}s")
        print(f"        Retrieval: Fused MRR={ep_va_mrr:.4f} | Local MRR={ep_va_local_mrr:.4f} | Global MRR={ep_va_global_mrr:.4f}")
        
        # Chon checkpoint theo MRR retrieval tren toan bo gallery validation; val loss la tie-breaker.
        if ep_va_mrr > best_val_mrr or (abs(ep_va_mrr - best_val_mrr) < 1e-4 and ep_va_tot < best_val_loss):
            best_val_mrr = ep_va_mrr
            best_val_loss = ep_va_tot
            torch.save({
                'run_config': RUN_CONFIG,
                'run_name': RUN_NAME,
                'train_songs': train_dataset.songs,
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_val_loss': best_val_loss,
                'best_val_mrr': best_val_mrr,
                'history': history
            }, finetuned_ckpt_path)
            epochs_without_improvement = 0
            print(f"       -> [NEW BEST] Epoch {epoch:02d} (Val Retrieval MRR: {best_val_mrr:.4f}, Val Loss: {best_val_loss:.4f})")
        else:
            epochs_without_improvement += 1

        resume_payload = {
            'loader_rng_state': loader_generator.get_state(),
            'epochs_without_improvement': epochs_without_improvement,
            'train_songs': train_dataset.songs,
            'run_config': RUN_CONFIG, 'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': amp_scaler.state_dict(), 'history': history,
            'best_val_loss': best_val_loss, 'best_val_mrr': best_val_mrr,
            'python_rng_state': random.getstate(), 'numpy_rng_state': np.random.get_state(),
            'torch_rng_state': torch.get_rng_state(),
            'cuda_rng_state': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        }
        torch.save(resume_payload, last_ckpt_path)

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print('[EARLY STOP] Validation did not improve.')
            break

    if os.path.exists(finetuned_ckpt_path):
        best_data = torch.load(finetuned_ckpt_path, map_location='cpu', weights_only=False)
        model.load_state_dict(best_data['model_state_dict'])
        model.eval()
    print("=" * 105)
    print("[+] Huấn luyện Fine-Tuning hoàn tất; model đã được nạp lại từ best checkpoint.")
    print(f"[+] Checkpoint tối ưu nhất (Val Retrieval MRR: {best_val_mrr:.4f}, Val Loss: {best_val_loss:.4f}) đã lưu tại:\n    -> {finetuned_ckpt_path}")

### 6.4 Truc quan hoa duong cong suy giam ham mat mat va metric truy van (Loss Curves & MRR)
Ve do thi 4 tieu chi theo epoch: Cosine Triplet Loss, Domain Adversarial Loss, Local Alignment Loss, va Validation MRR Metric.

In [ ]:
epochs = range(1, len(history['train_total_loss']) + 1)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 9), dpi=120)

# Retrieval objectives have the same terms on train and validation; LAC is intentionally separate.
ax1.plot(epochs, history['train_retrieval_loss'], marker='o', color='#2c3e50', linewidth=2.0, label='Train Retrieval Objective')
ax1.plot(epochs, history['val_total_loss'], marker='s', color='#e74c3c', linewidth=2.0, linestyle='--', label='Val Retrieval Objective')
ax1.set_title('1. Comparable Retrieval Objective', fontsize=11, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(True, linestyle=':', alpha=0.6); ax1.legend()

ax2.plot(epochs, history['train_lac_loss'], marker='o', color='#8e44ad', linewidth=2.0, label='Train LAC Cross-Entropy')
ax2.plot(epochs, history['train_prototype_loss'], color='#d35400', linewidth=1.8, label='Prototype Diversity')
ax2.set_title('2. Train-Only LAC Objectives', fontsize=11, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.grid(True, linestyle=':', alpha=0.6); ax2.legend()

ax3.plot(epochs, history['train_triplet_loss'], color='#27ae60', linewidth=2.0, label='Train Global Triplet')
ax3.plot(epochs, history['val_triplet_loss'], color='#16a085', linewidth=1.8, linestyle='--', label='Val Global Triplet')
ax3.plot(epochs, history['train_lat_loss'], color='#2980b9', linewidth=2.0, label='Train LAT')
ax3.plot(epochs, history['val_lat_loss'], color='#c0392b', linewidth=1.8, linestyle='--', label='Val LAT')
ax3.set_title('3. Global and Local Retrieval Losses', fontsize=11, fontweight='bold')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Loss'); ax3.grid(True, linestyle=':', alpha=0.6); ax3.legend()

if history.get('val_mrr'):
    ax4.plot(epochs, history['val_mrr'], marker='d', color='#8e44ad', linewidth=2.5, label='Selection MRR')
    ax4.plot(epochs, history['val_local_mrr'], color='#16a085', linewidth=1.8, label='Local MRR')
    ax4.plot(epochs, history['val_global_mrr'], color='#d35400', linewidth=1.8, label='Global MRR')
    ax4.set_title('4. Validation Retrieval Metric', fontsize=11, fontweight='bold')
    ax4.set_xlabel('Epoch'); ax4.set_ylabel('MRR'); ax4.grid(True, linestyle=':', alpha=0.6); ax4.legend()

plt.tight_layout()
loss_chart_path = os.path.join(OUTPUT_DIR, "loss_curves.png")
plt.savefig(loss_chart_path, dpi=150)
plt.show()
print(f"[+] Đã lưu biểu đồ loss tách biệt và MRR tại: {loss_chart_path}")


## Test retrieval and Kaggle checkpoint comparison


In [ ]:
# Test is evaluated only after model selection on validation.
# Protocol: test-only reference gallery, all chunks, exact local/global scoring;
# no FAISS gating or DTW. Selection MRR is the mean of local/global MRR,
# whereas fused ranking below averages their scores before ranking.
import json
import gc

@torch.no_grad()
def evaluate_test_retrieval(eval_model, dataset, label):
    eval_model.eval()
    gallery = validation_gallery_paths(dataset)
    song_ids = sorted(gallery)
    song_index = {song: i for i, song in enumerate(song_ids)}
    encoded = []
    for song in tqdm(song_ids, desc=label + ' gallery'):
        cqts = [chunk for path in gallery[song] for chunk in dataset._get_raw_cqts(path)]
        if not cqts:
            raise RuntimeError(f'Missing reference: {song}')
        encoded.append(_encode_cqt_list(eval_model, cqts))
    globals_list = [g for g, _ in encoded]
    locals_list = [l.reshape(-1, l.shape[-1]) for _, l in encoded]
    global_padded = nn.utils.rnn.pad_sequence(globals_list, batch_first=True)
    local_padded = nn.utils.rnn.pad_sequence(locals_list, batch_first=True)
    global_mask = torch.arange(global_padded.shape[1])[None, :] < torch.tensor([len(g) for g in globals_list])[:, None]
    local_mask = torch.arange(local_padded.shape[1])[None, :] < torch.tensor([len(l) for l in locals_list])[:, None]
    rows = []
    started = time.perf_counter()
    for song, fragment, path in tqdm(dataset.humming_samples, desc=label + ' queries'):
        g, l = _encode_cqt_list(eval_model, dataset._get_raw_cqts(path))
        local_scores = _vectorized_local_maxmean(l.to(device).reshape(-1, EMBEDDING_DIM), local_padded, local_mask)
        global_scores = _vectorized_global_chunk_scores(g.to(device), global_padded, global_mask)
        modes = {'local': local_scores, 'global': global_scores,
                 'fused': 0.5 * (local_scores + global_scores)}
        row = {'song': song, 'fragment': fragment, 'query': path}
        for mode, scores in modes.items():
            rank, ties = _average_tie_rank(scores, song_index[song])
            row[mode + '_rank'] = rank
            row[mode + '_ties'] = ties
            row[mode + '_top1_song'] = song_ids[int(scores.argmax())]
        rows.append(row)
    elapsed = time.perf_counter() - started
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError('No test queries.')
    result = {'label': label, 'queries': len(frame), 'gallery_songs': len(song_ids),
              'protocol': 'test_only_all_chunks_exact_maxmean_v4_average_ties',
              'query_wall_seconds': elapsed, 'ms_per_query_including_cqt_io': 1000 * elapsed / len(frame)}
    for mode in ('local', 'global', 'fused'):
        ranks = frame[mode + '_rank'].to_numpy()
        result[mode + '_mrr'] = float(np.mean(1.0 / ranks))
        # Expected Recall@K under uniform ordering within a tied score group.
        ties = frame[mode + '_ties'].to_numpy()
        greater = ranks - 1.0 - 0.5 * (ties - 1)
        for k in (1, 5, 10):
            result[f'{mode}_recall_at_{k}'] = float(np.clip((k - greater) / ties, 0, 1).mean())
    frame.to_csv(Path(OUTPUT_DIR) / (label + '_test_queries.csv'), index=False)
    return result

test_dataset = ByteHumCachedTripletDataset(TEST_DIR, prep, cqt_ext, is_training=False)
if set(test_dataset.songs) & (set(train_dataset.songs) | set(val_dataset.songs)):
    raise RuntimeError('Test/train/val song leakage.')
if not Path(finetuned_ckpt_path).is_file():
    raise FileNotFoundError(finetuned_ckpt_path)
best_data = torch.load(finetuned_ckpt_path, map_location='cpu', weights_only=False)
if best_data['run_config'] != RUN_CONFIG or best_data['train_songs'] != train_dataset.songs:
    raise ValueError('Best checkpoint does not match this local run.')
model.load_state_dict(best_data['model_state_dict'], strict=True)
results = [dict(evaluate_test_retrieval(model, test_dataset, 'local_4gb'),
                checkpoint=finetuned_ckpt_path, epoch=best_data['epoch'],
                validation_selection_mrr=best_data['best_val_mrr'])]

kaggle_checkpoint = STAGE2_ROOT / 'checkpoints' / 'conformer_bytehum_full_d128_timegrid_lat_v4_best.pth'
if LOCAL_COMPARE_KAGGLE and kaggle_checkpoint.is_file():
    comparison = torch.load(kaggle_checkpoint, map_location='cpu', weights_only=False)
    cfg = comparison.get('run_config', {})
    if cfg.get('pipeline_version') != 4 or cfg.get('embedding_dim') != EMBEDDING_DIM:
        raise ValueError('Kaggle checkpoint pipeline/embedding mismatch.')
    if comparison.get('train_songs') != train_dataset.songs:
        raise ValueError('Kaggle checkpoint train-song mapping mismatch.')
    model.load_state_dict(comparison['model_state_dict'], strict=True)
    results.append(dict(evaluate_test_retrieval(model, test_dataset, 'kaggle_v4'),
                        checkpoint=str(kaggle_checkpoint), epoch=comparison.get('epoch')))
    model.load_state_dict(best_data['model_state_dict'], strict=True)
elif LOCAL_COMPARE_KAGGLE:
    print('Comparison skipped: Kaggle checkpoint not found:', kaggle_checkpoint)

with open(Path(OUTPUT_DIR) / 'test_metrics.json', 'w', encoding='utf-8') as stream:
    json.dump({'run_config': RUN_CONFIG, 'results': results}, stream, indent=2)
display(pd.DataFrame(results).set_index('label'))
print('Results:', OUTPUT_DIR)
